<!-- notebook-header -->
<a id="top"></a>

# 3. Two Pointers, Sliding Window & Strings

*LeetCode Playbook · notebook 3 of 12* · Two fingers on a sequence, a window that crawls, and hand-written string parsing.

**In this notebook:** [Two Pointers](#s05) · [Sliding Window](#s06) · [Strings](#s20)

← [Arrays, Hashing & Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Stacks, Queues & Linked Lists](04_Stacks_Queues_Linked_Lists.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s05"></a>

## Two Pointers

> Two fingers on the array. Each step looks at the items under the fingers and moves one finger, and every move is a small *proof*: it throws away a whole family of candidates that cannot win. About n moves replace the n² pairs of the brute force.

**Reach for it when** the input is sorted, or you may sort it, and the question is about pairs or triplets: "sum to target", "how many pairs", "all unique triplets". Or when you compare mirror positions, as in palindromes and reversing. Or when an array must be rewritten **in place** with O(1) extra space, which means inside the array you were given and with no second array: "remove", "move the zeros", "dedupe", "partition into groups". Or when two sorted inputs must be merged or compared. Or when the answer is the area between two walls: a container, trapped rain water.

**In this repo:** `two_pointers/` (10 problems) · bank: `practice/simple/06_three_sum.py`, `practice/simple/07_container_with_most_water.py`, `practice/simple/08_trapping_rain_water.py` · basics: `practice/simple/basics/strings/02_two_pointer_palindromes_and_reverse_words.py`, `practice/simple/basics/sorting/03_quick_sort.py` (partition is a read / write pass), `practice/simple/basics/sorting/02_merge_sort.py` (merge is one finger per sorted list) · the same idea elsewhere: `linked_list/remove_nth_from_end.py` (two pointers with a fixed gap, [Linked Lists](04_Stacks_Queues_Linked_Lists.ipynb#s10))

### The picture

The brute force looks at every pair, so start by drawing every pair. Write them down as a table: one row per left item, one column per right item. The two fingers walk this table from its top-right corner, and the drawing shows why they never have to visit most of it.

```text
nums = [1, 2, 4, 6, 8, 9], target = 12       (row = left item, column = right item)

        2    4    6    8    9
  1     .    .    .    .   10 <     1 + 9 too small, and 9 was the biggest partner left: row 1 is dead
  2          .    .    .   11 <     the same for 2
  4               .   12 = 13 >     4 + 9 too big, and 4 was the smallest partner left: column 9 is dead;
  6                    .    .       then 4 + 8 = 12: found
  8                         .
```

Why it is fast: each comparison kills a whole row or a whole column of the table, so the walk is a staircase from the top-right corner: at most n − 1 steps instead of n² pairs. The answer is never killed, because only rows and columns that are proven hopeless go.

Four shapes cover almost every problem. The first two become the templates below; the other two are variations of them.

```text
opposite ends     L ->               <- R     pairs in sorted data, palindromes, containers, rain water
same direction    W ->  R ->                  read / write: keep some items, in place (W never passes R)
two inputs        i -> in a,  j -> in b       merge, subsequence, intersections, compare
three regions     low, mid, high              Dutch flag: [ 0s | 1s | unknown | 2s ]
```

### From idea to code

**The idea in one sentence:** *opposite ends: the comparison of the two ends tells you which one can never be part of a better answer, so move that finger inward until they meet. Same direction: read every item once, and copy each keeper to slot `write`.*

For a pair in sorted data, the **State** is two indices, `lo` and `hi`, and their **Definition** is one sentence: the candidates still alive are `nums[lo..hi]`. The **Invariant**, true after every step, is that if an answer exists, both of its indices are still inside `[lo, hi]`. A **Step** compares the two ends: too small, and `lo += 1`; too big, and `hi -= 1`. Each move drops only an item that the comparison has proven hopeless, so the invariant survives every move.

The **Record** is the hit, or every step when the problem keeps a best so far. **Init** puts the fingers on the two ends, `lo, hi = 0, len(nums) - 1`, after a sort if sorting is allowed. The loop runs `while lo < hi`, because a pair needs two different items, and the **Return** is the hit, or "not found" once the fingers meet. When equal values must not be reported twice, a finger steps over copies of the value it just used: `while lo < hi and nums[lo] == nums[lo - 1]: lo += 1`.

For a read / write pass, the state is one index, `write`, the next free slot, while `read` scans every item. The definition is that `nums[:write]` is the finished output, and the invariant says what it holds: the keepers of `nums[:read]`, in order. That has a consequence worth saying out loud: `write` never passes `read`, so nothing unread is ever overwritten. A step looks at `nums[read]`; if it is a keeper, `nums[write] = nums[read]` and `write += 1`, and that copy is also the record. `write` starts at 0, and it is returned as the new length.

```text
read / write:   [ kept  kept  kept | junk  junk | unread  unread ]
                                     ^            ^
                                   write        read
```

Every opposite-ends problem is one comparison and a move that the comparison justifies. In Two Sum II (167), the pair in a sorted array that sums to the target, the comparison is `a[lo] + a[hi]` against the target: too small kills the row and `lo` moves, too big kills the column and `hi` moves, and the hit is the record. 3Sum Closest (16), the triplet sum nearest the target, makes the same moves and records the best distance on every step.

Counting problems record in the move itself. Count Pairs Whose Sum is Less than Target (2824) asks how many pairs sum below the target, and 3Sum Smaller (259) asks the same for triplets: a pair that fits counts its whole row, `count += hi - lo`, before `lo` moves; otherwise `hi` moves. Valid Triangle Number (611), how many triples of lengths can be the sides of a triangle, compares `a[lo] + a[hi]` with the longest side `a[k]`: bigger counts the row and moves `hi`, otherwise `lo` moves.

When heights are compared, the move settles something for good. Container With Most Water (11) asks for the two walls that hold the most water: compare `h[lo]` with `h[hi]`, record the area on every step, and move the shorter wall, because it has met its best partner. Trapping Rain Water (42) asks how much water the whole skyline holds: the same comparison tells which side's water level is already known, so settle that bar and move on.

Three more fit the shape without a target at all. Boats to Save People (881), the fewest boats when each boat carries at most two people under a weight limit, compares the lightest plus the heaviest with the limit: the heaviest always leaves, and the lightest joins it if they fit, one boat per step. Valid Palindrome (125), whether a text reads the same backwards once punctuation and case are ignored, compares `s[lo]` with `s[hi]`: junk moves that finger alone, a match moves both, and a mismatch returns `False`.

The third, Squares of a Sorted Array (977), wants the squares of a sorted array, still sorted. It compares `abs(a[lo])` with `abs(a[hi])` and writes the bigger square to the back of the output, so the output fills from its end while the fingers close in.

The template solves Two Sum II with 0-based indices: in `[1, 2, 4, 6, 8, 9]` with target 12 the pair is `[2, 4]`, the items 4 and 8. A hit is reported before any finger moves past it, so RECORD comes before the STEPs. The second function is Remove Element (27), which drops every copy of one value in place and returns the new length: `[3, 2, 2, 3, 4]` without 3 keeps `[2, 2, 4]`. `read` visits every item once, and copying a keeper into slot `write` is both the step and the record.

In [ ]:
def pair_with_sum_sorted(nums, target):      # nums sorted ascending
    lo, hi = 0, len(nums) - 1                # STATE + INIT: the candidates still alive are nums[lo..hi]
    while lo < hi:                           # a pair needs two different items
        total = nums[lo] + nums[hi]
        if total == target:
            return [lo, hi]                  # RECORD
        if total < target:
            lo += 1                          # STEP: nums[lo] is too small even with the biggest partner left
        else:
            hi -= 1                          # STEP: nums[hi] is too big even with the smallest partner left
    return []                                # RETURN: no pair


def remove_value(nums, val):                 # keep every item except val, in place
    write = 0                                # STATE + INIT: nums[:write] = the keepers of nums[:read], in order
    for read in range(len(nums)):
        if nums[read] != val:                # a keeper
            nums[write] = nums[read]         # STEP + RECORD: copy it into the next free slot
            write += 1
    return write                             # RETURN: the new length


print(pair_with_sum_sorted([1, 2, 4, 6, 8, 9], 12))   # [2, 4]
arr = [3, 2, 2, 3, 4]
n_kept = remove_value(arr, 3)
print(n_kept, arr[:n_kept])                            # 3 [2, 2, 4]

**Try it**
- Change `while lo < hi` to `while lo <= hi` and run `pair_with_sum_sorted([2, 5, 9], 10)`: `[1, 1]`. The 5 was used twice.
- Run `pair_with_sum_sorted([4, 1, 3], 4)` on unsorted input: `[]`, although 1 + 3 = 4. The moves are only proofs when the data is sorted.
- Replace the body of `remove_value` with `nums = [x for x in nums if x != val]` and `return len(nums)`, then run it on `arr = [3, 2, 2, 3]`: it returns 2, but `arr` is untouched and `arr[:2]` is `[3, 2]`. The assignment only rebinds the local name.

### Watch it work

The same walk, printed one comparison per line, with `L` and `R` drawn under the items they point at. Seeing the staircase once makes the moves easy to trust: for `[1, 2, 4, 6, 8, 9]` and target 12, two sums are too small and one is too big before 4 + 8 hits.

In [ ]:
def trace_pair(nums, target):
    lo, hi = 0, len(nums) - 1
    print("nums " + "".join(f"{x:>4}" for x in nums))
    while lo < hi:
        total = nums[lo] + nums[hi]
        marks = "".join(f"{'L' if i == lo else 'R' if i == hi else '':>4}" for i in range(len(nums)))
        if total == target:
            print(f"     {marks}   {nums[lo]} + {nums[hi]} = {total}: found [{lo}, {hi}]")
            return
        if total < target:
            print(f"     {marks}   {nums[lo]} + {nums[hi]} = {total} < {target}: drop {nums[lo]} (lo += 1)")
            lo += 1
        else:
            print(f"     {marks}   {nums[lo]} + {nums[hi]} = {total} > {target}: drop {nums[hi]} (hi -= 1)")
            hi -= 1
    print("     no pair")


trace_pair([1, 2, 4, 6, 8, 9], 12)

**Try it**
- Run `trace_pair([1, 2, 4, 6, 8, 9], 7)`: this time R does the walking (9 and 8 are dropped), then 1 + 6 hits.
- Run `trace_pair([1, 2, 4, 6, 8, 9], 30)`: every sum is too small, so L walks all the way to R and the trace ends with "no pair".
- Match each line to a row or a column of the pair table in the picture: every "drop" kills one of them.

### Where it goes wrong

The bugs of this technique are a finger that lands one slot off, or a finger that moves for a reason the data does not prove. Each trap comes with its fix and a tiny input that exposes it.

1. **`<=` for pairs.** With `while lo <= hi` the fingers can meet on one item: `pair_with_sum_sorted([2, 5, 9], 10)` returns `[1, 1]`. Pairs need `lo < hi`. `<=` is right only when the middle item needs its own visit, as in the Dutch flag, whose `while mid <= high` must still look at the last unknown item. Squares of a Sorted Array needs that visit too, and its `for write` loop gives it by running exactly n times.
2. **Unsorted input.** Opposite ends works only because the order proves which side is hopeless: `[4, 1, 3]` with target 4 finds nothing. Sort first (carrying the original indices if they are the answer), or use a hash map.
3. **Moving the taller wall.** In Container With Most Water the *shorter* wall is the one that can never do better; moving the taller one gives 3 instead of 6 on `[1, 3, 2, 3]`.
4. **3Sum bookkeeping.** Skip an anchor equal to the *previous* one (`nums[i] == nums[i - 1]`); comparing with the *next* one throws away `[-1, -1, 2]`. After recording a triplet, move both fingers, or the loop records it forever.
5. **Dutch flag: advancing `mid` after a swap with `high`.** The value that came from the right has not been looked at yet: `[1, 2, 2, 0]` ends as `[1, 0, 2, 2]`.
6. **Looking back into the input instead of the output.** In Remove Duplicates II compare with `nums[write - 2]`, not `nums[read - 2]`: that slot may already be overwritten, so `[1, 1, 1, 2, 2]` keeps 3 items instead of 4.
7. **Unguarded skip loops.** `while not s[lo].isalnum(): lo += 1` runs off the end of `".,"` (`IndexError`). Guard every inner loop with `lo < hi`, or move one finger per iteration with `if / elif`.
8. **Rebinding instead of writing in place.** `nums = [x for x in nums if x != val]` builds a new list and points the local name at it; the caller's list does not change, so `[3, 2, 2, 3]` without 3 reports 2 kept items while still holding `[3, 2, 2, 3]`. Write through `nums[write]`, or use `nums[:] = ...` if extra space is allowed.

### Edge cases to say out loud

Empty input · one item · two items · all items equal (dedupe!) · no answer · negatives · the fingers meeting on one item · already sorted, or sorted backwards. Say them before you code; the asserts below check each one that applies to the two templates.

In [ ]:
assert pair_with_sum_sorted([], 5) == []
assert pair_with_sum_sorted([5], 10) == []                  # one item can't pair with itself
assert pair_with_sum_sorted([2, 5, 9], 10) == []            # 5 + 5 would reuse an item
assert pair_with_sum_sorted([3, 3], 6) == [0, 1]            # equal values, two items
assert pair_with_sum_sorted([-3, -1, 2, 7], 1) == [1, 2]    # negatives
all_sevens = [7, 7, 7]
assert remove_value(all_sevens, 7) == 0                     # everything removed
keep_all = [1, 2]
assert remove_value(keep_all, 9) == 2 and keep_all == [1, 2]   # nothing removed
assert remove_value([], 1) == 0
print("edge cases pass")

**Try it**
- LeetCode 167 wants 1-based indices: return `[lo + 1, hi + 1]` and check that `[2, 7, 11, 15]` with target 9 gives `[1, 2]`.
- Predict, then add: `assert pair_with_sum_sorted([1, 1, 1, 1], 2) == [0, 3]` (the outermost pair is found first).
- Add `assert pair_with_sum_sorted([1, 5, 10], 11) == [0, 2]`: an answer that uses both ends is found on the very first comparison.

### Variations

The opposite-ends problems above use the template as it is; the variations change its shape, with more fingers, a second input or a finger that writes. The table is the overview, in the order of the cells that follow it. The fixed gap is the one row coded elsewhere, in [Linked Lists](04_Stacks_Queues_Linked_Lists.ipynb#s10).

| Variation | What changes from the template | Problems |
|---|---|---|
| **k-sum** | sort; fix one item (or k − 2 items); two-pointer the rest; skip equal neighbours | 3Sum (15): every unique triplet that sums to 0; 4Sum (18): the same for four numbers |
| **Count pairs** | when a pair fits, every partner between the fingers fits too: `count += hi - lo` | Count Pairs Whose Sum is Less than Target (2824), 3Sum Smaller (259), Valid Triangle Number (611) |
| **Heights** | record on every step; the finger on the lower wall or bar moves, and that side is settled | Container With Most Water (11), Trapping Rain Water (42) |
| **Ends into an output** | compare absolute values; the bigger square goes to the back of the output | Squares of a Sorted Array (977) |
| **Mirror positions** | junk under a finger moves that finger alone; the first mismatch decides | Valid Palindrome (125) |
| **Two inputs, one finger each** | merge from the back (88); advance the pattern finger only on a match (392); advance the list whose interval ends first (986); walk from the end, skipping deleted chars (844) | Merge Sorted Array (88): merge one sorted array into another, in place; Is Subsequence (392): can `s` be read inside `t`; Interval List Intersections (986): the overlaps of two interval lists; Backspace String Compare (844): equal once each `#` deletes a character |
| **Read / write compaction** | `write` trails `read`; copy (or swap) the keepers | Remove Element (27); Move Zeroes (283): the zeros to the back, the rest in order |
| **Look-back dedupe** | keep `nums[read]` unless it equals `nums[write - k]` | Remove Duplicates from Sorted Array (26): at most one copy; II (80): at most two |
| **Three pointers, four regions** | Dutch flag: `low`, `mid`, `high`; after a swap with `high`, `mid` stays | Sort Colors (75): sort 0s, 1s and 2s in one pass |
| **Fixed gap** | the front finger starts k ahead and both move together ([Linked Lists](04_Stacks_Queues_Linked_Lists.ipynb#s10)) | Remove Nth Node From End of List (19): unlink the n-th node from the end |
| *Second pass:* **two strings with a bookmark** | one finger per string; on a mismatch, jump back to the last `*` | Wildcard Matching (44): does a pattern with `?` and `*` match the whole text |

The first variation is k-sum. Once a pair is a staircase walk, a triplet is that walk run once per *anchor*, the item you hold fixed. 3Sum asks for every unique triplet that sums to 0: `[-1, 0, 1, 2, -1, -4]` gives `[-1, -1, 2]` and `[-1, 0, 1]`. Sort, fix `nums[i]`, and two-pointer the rest for `-nums[i]`. Sorting also puts equal values side by side, so duplicates are skipped by comparing with a neighbour instead of collecting a set, and 4Sum, the same question for four numbers, wraps one more loop around it.

In [ ]:
def three_sum(nums):
    nums = sorted(nums)
    n, out = len(nums), []
    for i in range(n - 2):
        if nums[i] > 0:
            break                            # the smallest of the three is positive: no zero sum is left
        if i > 0 and nums[i] == nums[i - 1]:
            continue                         # same anchor as last time: same triplets
        lo, hi = i + 1, n - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if total < 0:
                lo += 1                      # STEP
            elif total > 0:
                hi -= 1                      # STEP
            else:
                out.append([nums[i], nums[lo], nums[hi]])   # RECORD
                lo, hi = lo + 1, hi - 1      # STEP: a new triplet needs BOTH values to change
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1                  # skip copies of the value just used
    return out


print(three_sum([-1, 0, 1, 2, -1, -4]))   # [[-1, -1, 2], [-1, 0, 1]]
print(three_sum([0, 0, 0, 0]))            # [[0, 0, 0]]

**Try it**
- Replace the anchor check with `if nums[i] == nums[i + 1]: continue` and run `three_sum([-1, -1, 2])`: `[]`. The first −1 was skipped while it was still needed.
- Delete the inner `while ... nums[lo] == nums[lo - 1]` loop and run `three_sum([-2, 0, 0, 2, 2])`: `[[-2, 0, 2], [-2, 0, 2]]`, a duplicate.
- There is no skip loop for `hi`. Print `i, lo, hi, total` inside the `while` for `[-2, 0, 1, 1, 2, 2]`: after the first hit, `hi` sits on a copy of 2, the total is 1, and `total > 0` moves it on by itself.

The walk can count instead of find. Count Pairs Whose Sum is Less than Target asks how many pairs sum below the target: `[-1, 1, 2, 3, 1]` with target 2 has 3 such pairs. When `nums[lo] + nums[hi]` fits, `nums[lo]` fits with every partner between the fingers too, because they are all smaller than `nums[hi]`, so one comparison counts a whole row of the pair table and `lo` moves on. When it does not fit, `hi` is hopeless and moves.

In [ ]:
def count_pairs_below(nums, target):         # pairs i < j with nums[i] + nums[j] < target (2824, 259)
    nums = sorted(nums)
    lo, hi, count = 0, len(nums) - 1, 0
    while lo < hi:
        if nums[lo] + nums[hi] < target:
            count += hi - lo                 # RECORD: nums[lo] fits with EVERY partner in lo+1..hi
            lo += 1                          # STEP: that whole row is counted
        else:
            hi -= 1                          # STEP: nums[hi] is too big even with the smallest partner
    return count


print(count_pairs_below([-1, 1, 2, 3, 1], 2))   # 3

**Try it**
- Replace `count += hi - lo` with `count += 1`: `count_pairs_below([-1, 1, 2, 3, 1], 2)` gives 1 instead of 3. One comparison proved a whole row, so it must count a whole row.
- Print `lo, hi` whenever the pair fits: for that input the only hit is `lo = 0, hi = 3` on the sorted list `[-1, 1, 1, 2, 3]`, and it adds 3 pairs at once (−1 with 1, 1 and 2).
- Write Valid Triangle Number (611), the same move from the other side: sort, fix the longest side `sides[k]` from the right, and run the fingers on `sides[:k]`; when `sides[lo] + sides[hi] > sides[k]`, do `count += hi - lo` and `hi -= 1`, else `lo += 1`. Check that `[2, 2, 3, 4]` gives 3.

Heights change what a move proves: the finger leaves behind a wall or a bar that is settled for good. Container With Most Water asks for the two walls that hold the most water: in `[1, 8, 6, 2, 5, 4, 8, 3, 7]` the walls of height 8 and 7, seven apart, hold 49. The area is `min(left, right) × width`. The shorter wall caps the height, and any narrower container that keeps it is no better, so the shorter wall has already met its best partner: drop it and move that finger inward.

Trapping Rain Water asks how much water the whole skyline holds: `[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]` holds 6. The water above one bar is `min(tallest on its left, tallest on its right) − bar`:

```text
height = [3, 0, 2, 0, 4]

            #          water above each bar = min(max on its left, max on its right) - bar
#  ~  ~  ~  #
#  ~  #  ~  #          = 3 + 1 + 3 = 7
#  ~  #  ~  #
3  0  2  0  4
```

The two fingers settle one bar per step, always the bar under the lower finger, and the bars between the fingers never need to be seen for that. A finger only ever leaves the lower of the two current bars, so the taller current bar is at least as tall as every bar passed so far. When `height[lo] < height[hi]`, that makes `left_max` at most `height[hi]`, a bar standing to the right of `lo`. So the level at `lo` is exactly `left_max`, and the water above `lo` is settled.

Both functions below walk the same two fingers inward. `max_area` measures the container between them and drops the shorter wall, which gives 49 for the walls above. `trap` fills the bar under the lower finger up to the tallest bar passed on its own side, then moves that finger on, which gives 6 for the skyline above.

In [ ]:
def max_area(height):
    lo, hi, best = 0, len(height) - 1, 0
    while lo < hi:
        best = max(best, min(height[lo], height[hi]) * (hi - lo))   # RECORD every pair we look at
        if height[lo] < height[hi]:
            lo += 1                          # STEP: the shorter wall can't do better with a closer partner
        else:
            hi -= 1
    return best


def trap(height):
    lo, hi = 0, len(height) - 1
    left_max = right_max = water = 0         # STATE: the tallest bar passed so far from each side
    while lo < hi:
        if height[lo] < height[hi]:
            left_max = max(left_max, height[lo])
            water += left_max - height[lo]   # RECORD: left_max <= height[hi], so left_max sets the level at lo
            lo += 1                          # STEP
        else:
            right_max = max(right_max, height[hi])
            water += right_max - height[hi]  # RECORD: right_max <= height[lo], so right_max sets the level at hi
            hi -= 1                          # STEP
    return water


print(max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))         # 49
print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))    # 6

**Try it**
- In `max_area`, flip `<` to `>` so the taller wall moves, and run `max_area([1, 3, 2, 3])`: 3 instead of 6.
- In the left branch of `trap`, swap the two lines (add water before updating `left_max`) and run `trap([2, 0, 3])`: 0 instead of 2. The first bar added 0 − 2 = −2 units of "water".
- Print `lo, hi, left_max, right_max, water` at the top of the `trap` loop for `[4, 2, 0, 3, 2, 5]`: `hi` never moves (5 is the tallest bar, so the left side is always the lower one), and the water grows 0, 0, 2, 6, 7 before the last step brings it to 9.

The fingers can also produce an output in order. Squares of a Sorted Array asks for the squares of a sorted array, still sorted: `[-4, -1, 0, 3, 10]` becomes `[0, 1, 9, 16, 100]`. Squaring folds the array, so the biggest squares sit at the two ends and the smallest somewhere in the middle. Compare the ends by absolute value, write the bigger square into the last free slot of the output, and move that finger inward.

In [ ]:
def sorted_squares(nums):                    # nums sorted; returns the squares, sorted
    out = [0] * len(nums)
    lo, hi = 0, len(nums) - 1
    for write in range(len(nums) - 1, -1, -1):   # fill from the back: the biggest square first
        if abs(nums[lo]) > abs(nums[hi]):
            out[write] = nums[lo] ** 2       # RECORD
            lo += 1                          # STEP
        else:
            out[write] = nums[hi] ** 2
            hi -= 1
    return out


print(sorted_squares([-4, -1, 0, 3, 10]))   # [0, 1, 9, 16, 100]
print(sorted_squares([-7, -3, 2, 3, 11]))   # [4, 9, 9, 49, 121]

**Try it**
- Fill `out` from the front (`for write in range(len(nums))`) and run `[-4, -1, 0, 3, 10]`: `[100, 16, 9, 1, 0]`. The ends hold the biggest squares, so they belong at the back.
- Run `sorted_squares([-3, -2, -1])`: `[1, 4, 9]`. `lo` takes the first two squares; for the last item `lo == hi`, the `>` is False and the `else` branch takes it, so `hi` moves once.
- Change `>` to `>=` and rerun `[-7, -3, 2, 3, 11]`: the same output. On a tie both ends hold the same square, so either one may go to the back.

The same two fingers walk mirror positions. Valid Palindrome asks whether a text reads the same both ways when only letters and digits count and case is ignored: `"A man, a plan, a canal: Panama"` does, and so does `".,"`, which has nothing to compare. Its sibling Valid Palindrome II (680), which allows one deletion, is coded in [Strings](#s20). Here junk under a finger moves that finger alone, and the loop's own `lo < hi` keeps that skip from running off the end. The first mismatch of two lowercased characters decides the answer at once.

In [ ]:
def is_palindrome_clean(s):                  # 125: letters and digits only, case ignored
    lo, hi = 0, len(s) - 1
    while lo < hi:
        if not s[lo].isalnum():
            lo += 1                          # skip junk: move ONE finger, compare nothing
        elif not s[hi].isalnum():
            hi -= 1
        elif s[lo].lower() != s[hi].lower():
            return False                     # RECORD: a mismatch decides it
        else:
            lo, hi = lo + 1, hi - 1
    return True


print(is_palindrome_clean("A man, a plan, a canal: Panama"), is_palindrome_clean(".,"))   # True True

**Try it**
- Replace the `if/elif` skips with `while not s[lo].isalnum(): lo += 1` and its mirror for `hi`, then run `is_palindrome_clean(".,")`: `IndexError` (trap 7). The inner loop runs off the end, because nothing checks `lo < hi` inside it.
- Drop both `.lower()` calls and run `is_palindrome_clean("Aa")`: `False`. Case only stops mattering once both sides are lowered.
- Change `isalnum` to `isalpha` in both skips and run `is_palindrome_clean("0P")`: `True` instead of `False`. Digits count as characters, so the `0` must be compared with the `P`.
- Print `lo, hi` just before `return False` and run `is_palindrome_clean("race a car")`: `3 5`. The `e` meets the `a` after `hi` has skipped one space.

With two sorted inputs, each gets its own finger, and the comparison decides which one moves. Is Subsequence (392) asks whether `s` can be read inside `t` by skipping letters of `t`, and advances the pattern finger only on a match. Interval List Intersections (986) asks for the overlaps of two sorted lists of intervals, and advances the list whose interval ends first. Backspace String Compare (844) asks whether two strings are equal after each `#` deletes the character before it, and walks both strings from the end, skipping the characters a later `#` deletes.

Merge Sorted Array (88) gets `a` with m sorted items followed by n spare slots and must merge `b` into it in place: `[1, 2, 3, 0, 0, 0]` and `[2, 5, 6]` become `[1, 2, 2, 3, 5, 6]`. Filling from the front would overwrite items of `a` before they are read; the spare slots are at the back, so fill from the back, taking the bigger of the two tails each time. A used-up input simply loses every comparison.

In [ ]:
def merge_into(a, m, b, n):                  # 88: a holds m sorted items + n spare slots; b holds n
    i, j = m - 1, n - 1                      # STATE: the last unmerged item of each input
    for write in range(m + n - 1, -1, -1):   # fill from the back: those slots are free
        if j < 0 or (i >= 0 and a[i] > b[j]):
            a[write] = a[i]                  # STEP + RECORD: the bigger tail goes last
            i -= 1
        else:
            a[write] = b[j]
            j -= 1
    return a


print(merge_into([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3))   # [1, 2, 2, 3, 5, 6]

**Try it**
- Drop `i >= 0 and` and run `merge_into([2, 0], 1, [1], 1)`: `[2, 2]`. Once `a` is used up, `a[-1]` quietly reads the last slot.
- Drop `j < 0 or` and run `merge_into([1], 1, [], 0)`: `IndexError`, because `b` is empty and `b[j]` with j = −1 fails.
- Merge from the front instead (`i = j = 0`, take the smaller head, `write` counting up): `[1, 2, 2, 2, 5, 6]`. `b`'s 2 overwrote the 3 before it was read.

The read / write template has its own small family, and in both members below `read` visits every item once while `write` marks where the next keeper goes. Move Zeroes asks to push every 0 to the back while the other items keep their order: `[0, 1, 0, 3, 12]` becomes `[1, 3, 12, 0, 0]`. It swaps instead of copying, so the zero that sat at `write` travels back to `read`, and the zeros collect at the end in the same pass.

Remove Duplicates from Sorted Array II asks to keep at most two copies of each value in a sorted array, in place: `[1, 1, 1, 2, 2, 3]` keeps 5 items, `[1, 1, 2, 2, 3]`. It copies an item unless it would be a third copy, and because the array is sorted, a third copy is exactly an item equal to the one two slots back in the output.

In [ ]:
def move_zeroes(nums):
    write = 0                                # STATE: nums[:write] = the non-zeros so far, in order
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]   # STEP: the zero at write moves back to read
            write += 1


def remove_extra_copies(nums, k=2):          # sorted nums: keep at most k copies of each value
    write = 0
    for read in range(len(nums)):
        if write < k or nums[read] != nums[write - k]:   # look back k slots into the OUTPUT
            nums[write] = nums[read]         # STEP + RECORD
            write += 1
    return write


zeros = [0, 1, 0, 3, 12]
move_zeroes(zeros)
print(zeros)                                 # [1, 3, 12, 0, 0]
dupes = [1, 1, 1, 2, 2, 3]
kept = remove_extra_copies(dupes)
print(kept, dupes[:kept])                    # 5 [1, 1, 2, 2, 3]

**Try it**
- Look back into the input instead: change the condition to `read < k or nums[read] != nums[read - k]` and run `remove_extra_copies([1, 1, 1, 2, 2])`: 3 items instead of 4. The slot it looks back at was already overwritten.
- Print `nums` after each swap in `move_zeroes([0, 1, 0, 3, 12])`: `[1, 0, 0, 3, 12]`, `[1, 3, 0, 0, 12]`, `[1, 3, 12, 0, 0]`. The block of zeros slides to the right.
- Copy instead of swapping (`nums[write] = nums[read]`) in `move_zeroes`: you get `[1, 3, 12, 3, 12]`, and a second loop would have to write the zeros.

With three kinds of item, one write finger is not enough. Sort Colors asks to sort an array of 0s, 1s and 2s in place, in one pass: `[2, 0, 2, 1, 1, 0]` becomes `[0, 0, 1, 1, 2, 2]`. The trick is named after the three bands of the Dutch flag: keep four regions, `[0s | 1s | unknown | 2s]`, and shrink the unknown one by one item per step.

```text
[ 0 0 | 1 1 | ? ? ? | 2 2 ]
        ^     ^   ^             nums[:low] are 0s, nums[low:mid] are 1s,
       low   mid high           nums[mid..high] are unknown, nums[high+1:] are 2s
```

`mid` looks at the first unknown item: a 0 is swapped down to `low`, a 1 stays, a 2 is swapped up to `high`. What comes back from `low` is a 1 that was already looked at, but what comes back from `high` is unknown, so after that swap `mid` stays put. [Sorting & Selection](05_Binary_Search_Sorting.ipynb#s23) runs the same loop around any pivot value.

In [ ]:
def sort_colors(nums):
    low, mid, high = 0, 0, len(nums) - 1     # STATE: the four regions above
    while mid <= high:                       # the unknown region nums[mid..high] is not empty
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low, mid = low + 1, mid + 1      # STEP: what came from low is a 1 (or low == mid): done with it
        elif nums[mid] == 1:
            mid += 1                         # STEP: a 1 is already in its region
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1                        # STEP: what came from high is unknown, so mid stays to look at it


colors = [2, 0, 2, 1, 1, 0]
sort_colors(colors)
print(colors)                                # [0, 0, 1, 1, 2, 2]

**Try it**
- Print `nums[:low], nums[low:mid], nums[mid:high + 1], nums[high + 1:]` at the top of the loop for `[2, 0, 2, 1, 1, 0]`: the four regions, with the unknown one shrinking by exactly one item per round (6 rounds).
- Add `mid += 1` after `high -= 1` and sort `[1, 2, 2, 0]`: `[1, 0, 2, 2]`. The 0 that came from the right was never looked at.
- With the same print, run `[2, 2, 2]` and `[0, 0, 0]`: only `high` moves in the first; `low` and `mid` move together in the second.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Wildcard Matching puts one finger on a text and one on a pattern. It asks whether a pattern with `?` for any one character and `*` for any run of characters, even none, matches the whole text: `"*a*b"` matches `"adceb"`, and `"a*c?b"` does not match `"acdcb"`. A plain character or `?` moves both fingers. A `*` is a safety net: bookmark it and let it match nothing for now.

On a mismatch, go back to the latest bookmark and let that star swallow one more character. Only the latest star matters, because anything an earlier star could swallow, the later one can swallow instead. The cost is O(|s|·|p|) in the worst case, because every mismatch can send the pattern finger back to the bookmark.

In [ ]:
def wildcard_match(s, p):                    # '?' = any one char, '*' = any run of chars (even none)
    i = j = 0
    star, mark = -1, 0                       # STATE: the last '*' in p, and where in s its match ends
    while i < len(s):
        if j < len(p) and p[j] in (s[i], "?"):
            i, j = i + 1, j + 1              # STEP: plain match, both fingers move
        elif j < len(p) and p[j] == "*":
            star, mark = j, i                # STEP: bookmark; the star matches nothing for now
            j += 1
        elif star != -1:
            mark += 1                        # FIX: on a mismatch the last star swallows one more char
            i, j = mark, star + 1
        else:
            return False                     # RETURN: a mismatch and no star to fall back on
    while j < len(p) and p[j] == "*":
        j += 1                               # leftover stars match the empty string
    return j == len(p)                       # RETURN


print(wildcard_match("adceb", "*a*b"), wildcard_match("acdcb", "a*c?b"))   # True False

**Try it**
- Delete the final `while` that skips leftover stars and run `wildcard_match("ab", "ab*")`: `False` instead of `True`.
- Print `i, j, star, mark` at the end of the mismatch branch for `("adceb", "*a*b")`: three mismatches, with `mark` at 2, 3 and 4 as the second star swallows `"d"`, `"dc"` and `"dce"`; then `b` matches.
- Count the rounds of the main loop for `s = "a" * 100` and `p = "*" + "a" * 50 + "b"`: 2,601. Double both lengths: 10,201. Four times the work, the O(|s|·|p|) worst case.

### Say it in the interview

The script is the picture in words: the brute force, the waste it repeats, the proof behind each move, and the cost.

> "Brute force checks every pair: O(n²). The array is sorted, so the sum of the two ends tells me which end is hopeless: if it's too small, the left number can't reach the target even with the largest partner left; if it's too big, the right one can't even with the smallest."

> "Each comparison deletes a whole row or column of the pair table, so it's at most n − 1 steps: O(n) time and O(1) space, plus O(n log n) if I have to sort first. For 3Sum I sort, fix the first number and run this on the rest: O(n²)."

Then point at each pointer move and say what it rules out ("everything in this row is too small"). For in-place problems, say the invariant out loud: "`nums[:write]` is the finished answer, and `write` never passes `read`." Follow-ups to have ready:

- *Count the pairs instead of finding one?* `count += hi - lo` when the pair fits, then `lo += 1`.
- *Closest sum instead of exact?* Record the best distance on every step; the moves stay the same.
- *Unsorted, and the original indices are the answer?* A hash map ([Arrays & Hashing](02_Arrays_Hashing_Prefix_Sums.ipynb#s03)), or sort `(value, index)` pairs.
- *k-sum in general?* Fix k − 2 items in nested loops, two pointers inside: O(n^(k−1)).

### Problem map

Every problem of this folder, where its file lives, and the one idea that cracks it.

| Problem | Where | Key insight |
|---|---|---|
| 3Sum | `two_pointers/three_sum.py` · `practice/simple/06_three_sum.py` | sort; fix nums[i]; two-pointer the rest; skip equal neighbours for the anchor and after each hit |
| Container With Most Water | `two_pointers/container_with_most_water.py` · `practice/simple/07_container_with_most_water.py` | start widest; the shorter wall can't do better with a closer partner, so move it inward |
| Move Zeroes | `two_pointers/move_zeroes.py` | write = next slot for a non-zero; swapping keepers forward pushes the zeros to the back |
| Remove Duplicates from Sorted Array II | `two_pointers/remove_duplicates_from_sorted_array_ii.py` | keep nums[read] unless it equals nums[write − 2]: look back into the output |
| Sort Colors | `two_pointers/sort_colors.py` | Dutch flag: low / mid / high regions; after swapping with high, mid stays |
| Squares of a Sorted Array | `two_pointers/squares_of_a_sorted_array.py` | the biggest square is at one of the two ends; fill the output from the back |
| Trapping Rain Water | `two_pointers/trapping_rain_water.py` · `practice/simple/08_trapping_rain_water.py` | move the finger at the lower bar: its running max is at most the other bar, so it sets the level |
| Two Sum II - Input Array Is Sorted | `two_pointers/two_sum_ii_input_array_is_sorted.py` | ends of a sorted array: too small → lo += 1, too big → hi −= 1; answer is 1-based |
| Valid Palindrome | `two_pointers/valid_palindrome.py` | compare mirror characters moving inward; skip non-alphanumerics without running off the end |
| Wildcard Matching | `two_pointers/wildcard_matching.py` | greedy match with a bookmark at the last `*`; on a mismatch that star swallows one more char |

### Self-check

1. In Two Sum II, when `nums[lo] + nums[hi]` is too small you move `lo`. Why can that never skip the answer?
<details><summary>Answer</summary><code>nums[hi]</code> is the biggest value still in play, and even it is not enough for <code>nums[lo]</code>. Every other partner left is smaller, so no pair that uses <code>nums[lo]</code> can reach the target. Dropping it removes only hopeless pairs: one dead row of the pair table.</details>

2. In Trapping Rain Water, why is the water above `lo` settled when `height[lo] < height[hi]`, although the bars between the fingers have not been seen yet?
<details><summary>Answer</summary>The level at <code>lo</code> is min(tallest bar on its left, tallest bar on its right). A finger only ever moves off the <em>lower</em> current bar, so no bar passed so far is taller than <code>height[hi]</code>, the taller current bar. That gives <code>left_max <= height[hi]</code>, and <code>height[hi]</code> stands right of <code>lo</code>: the minimum is <code>left_max</code>. Unseen bars between the fingers can only raise the right side, which leaves the minimum unchanged.</details>

3. In counting pairs below a target, why does one successful comparison add `hi - lo` pairs?
<details><summary>Answer</summary>The array is sorted, so every partner between <code>lo + 1</code> and <code>hi</code> is at most <code>nums[hi]</code>. If <code>nums[lo] + nums[hi]</code> is below the target, so is <code>nums[lo]</code> plus any of them: that is <code>hi - lo</code> pairs, the whole row, counted at once before <code>lo</code> moves on.</details>

<a id="s06"></a>

## Sliding Window

> A window `[left .. right]` crawls along the array like a caterpillar. Its front, `right`, takes one new item every step; its back, `left`, moves only to repair the window in longest problems or to tighten it in shortest ones. Each item enters once and leaves once, so the whole crawl is O(n).

[Two Pointers](#s05) moved two fingers toward each other, and every move was a proof that killed a row of pairs. A window is two fingers that both move right, and its proof is that a broken window stays broken: no longer window that contains it can be valid, so the back finger never has to move back.

**Reach for it when** the problem says *contiguous*, a substring or a subarray, and asks for the **longest**, **shortest**, or **number of** windows that satisfy a rule, and the rule is **monotone**: adding an item can never repair a broken window. Quick test: "can adding one more item make a bad window good again?" If yes, as for a sum with negative numbers, this tool does not apply; use [Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb#s04) instead.

**In this repo:** `sliding_window/` (13 problems) · bank: `practice/simple/09_longest_substring_without_repeating_characters.py`, `practice/simple/10_minimum_window_substring.py`, `practice/simple/11_sliding_window_maximum.py`, `practice/simple/12_longest_repeating_character_replacement.py`

### The picture

```text
index  0 1 2 3 4 5 6 7
s      a b c a b c b b         rule: no letter twice inside s[L..R]

       L   R                   "abc" valid -> record 3
       L     R                 R eats s[3]='a': "abca" has two a's -> broken
         L   R                 L lets go of the old 'a': "bca" valid -> record 3
           L   R               same story for 'b': "cab"
                ...
                 L   R         R eats s[7]='b': "cbb" -> broken
                     L=R       L lets go of 'c' AND 'b' (while, not if): "b"
```

**Why it is fast:** the brute force restarts from every `left` and re-reads the same letters. Two neighbouring windows share everything except one item at each end, so we *update* instead of *recompute*. `left` and `right` only move forward, each at most n steps: O(n) in total, even though there is a loop inside a loop.

**Why it is correct:** if `[left..right]` is broken, every window that starts at or before `left` and ends at or after `right` contains it, so it is broken too; that is what "monotone" buys. The start `left` is finished for good, which is why `left` never moves back. And once the window is repaired, every window `[i..right]` with `left ≤ i ≤ right` sits inside it and is valid too: `right - left + 1` valid windows end at `right`. Negative numbers break exactly this argument.

### From idea to code

**The idea in one sentence:** *the front eats one item; then the back moves until the window is the best it can be for this front, repaired for longest and as tight as possible for shortest, and the answer is recorded at the moment the window is known to be valid.*

Longest Substring Without Repeating Characters (3) asks for the longest stretch with no letter twice, and it is the template for every *longest* window. Its **State** is the back finger `left` plus a summary of the window, and the summary's **Definition** is `count[c]`, the copies of `c` in `s[left..right]`. The **Invariant** holds after the repair loop: the window is valid. A **Step** adds `s[right]` to the summary, and the **Fix** drops `s[left]` and moves `left` while the window is broken.

The **Record** comes after the fix, because only then is the window valid, and at that moment it is the longest valid window ending at `right`. **Init** is `left = 0`, an empty summary and `best = 0`, and the **Return** is `best`.

Minimum Size Subarray Sum (209) asks for the shortest subarray whose sum reaches a target, and it turns the loop inside out for every *shortest* window. Its summary is `total`, the sum of `nums[left..right]`, and its fix runs *while the window is valid*: record, then drop `nums[left]`, because a longer window with the same start is useless. After that loop the window is *not* valid, and every earlier start has had its shortest window recorded. Init is `best = math.inf`, and the return translates `inf` into 0, `""` or −1.

The step that freezes people is turning the rule into a summary that updates in O(1) and a `while` condition. "No repeated letter" keeps a `count` dict and is broken when `count[c] > 1`: only the new letter `c` can be the doubled one. "At most k distinct" keeps the same dict, deletes keys that fall to zero, and is broken when `len(count) > k`. That is Fruit Into Baskets (904), the longest stretch of trees holding at most two kinds of fruit, and Longest Substring with At Most K Distinct Characters (340).

Budget rules count what the window spends. Max Consecutive Ones III (1004), the longest run of 1s once at most k zeros are flipped, keeps `zeros` and is broken when `zeros > k`. Longest Repeating Character Replacement (424), the longest stretch that becomes one repeated letter after at most k replacements, keeps `count` and `max_count` and is broken when `(right - left + 1) - max_count > k`. Every rule so far records after the fix.

Shortest rules are *valid* conditions, and they record inside the fix. Minimum Size Subarray Sum, with no negative numbers, stays valid while `total >= target`. Minimum Window Substring (76), the shortest stretch of `s` that holds every letter of `t` with its count, keeps `window` and `formed` and stays valid while `formed == len(need)`. Counting rules record on every step: Subarray Product Less Than K (713), how many subarrays have a product below k, is broken when `prod >= k` and records `count += right - left + 1`.

The first function solves Longest Substring Without Repeating Characters: `"abcabcbb" → 3`, for `"abc"`. Its words become lines one by one: the front eats the next letter, `count[c] += 1`; while the window is broken, `while count[c] > 1:`, the back lets go, `count[s[left]] -= 1` and then `left += 1`, in that order; the length is `right - left + 1`. The second function solves Minimum Size Subarray Sum: `[2, 3, 1, 2, 4, 3]` with target 7 gives 2, for `[4, 3]`, recorded inside the loop while the window is still valid.

In [ ]:
def longest_without_repeat(s):
    count = defaultdict(int)                 # STATE: count[c] = copies of c in s[left..right]
    left, best = 0, 0                        # INIT
    for right, c in enumerate(s):
        count[c] += 1                        # STEP: the front eats s[right]
        while count[c] > 1:                  # FIX while broken (while, not if)
            count[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)   # RECORD: the window is valid here
    return best                              # RETURN


def shortest_with_sum_at_least(nums, target):     # non-negative numbers only
    total, left, best = 0, 0, math.inf       # STATE + INIT (inf = "not found yet")
    for right, x in enumerate(nums):
        total += x                           # STEP
        while left <= right and total >= target:  # FIX: valid, so tighten it
            best = min(best, right - left + 1)    # RECORD before breaking it
            total -= nums[left]
            left += 1
    return 0 if best == math.inf else best   # RETURN: translate "not found"


print(longest_without_repeat("abcabcbb"), longest_without_repeat("pwwkew"))   # 3 3
print(shortest_with_sum_at_least([2, 3, 1, 2, 4, 3], 7))                       # 2  ([4, 3])

**Try it**
- Change `while count[c] > 1:` to `if count[c] > 1:` and run `longest_without_repeat("abcbd")`. It answers 4 instead of 3: one step of `left` was not enough, so the broken window `"bcbd"` got recorded.
- Move the `best = max(...)` line *above* the `while` loop and rerun `"abcabcbb"`: you get 4, because you recorded `"abca"` before fixing it.
- In `shortest_with_sum_at_least`, move the `best = min(...)` line *below* `left += 1`: `[2, 3, 1, 2, 4, 3], 7` now answers 1, because it measured windows that were already broken.
- Feed it negatives: `shortest_with_sum_at_least([1, -1, 5, 1], 6)` says 4, but `[5, 1]` has length 2. Dropping the `1` lowered the sum, so the loop stopped before it could drop the `-1` that was in the way. The rule is not monotone any more.

### Watch it work

The trace runs the first template on `"abcabcbb"` and prints one line per step of `right`, with the window drawn under the string at its real position. Watch `left`: it only ever moves forward, sometimes by two letters at once, and `best` never shrinks.

In [ ]:
def trace_longest(s):
    count, left, best = defaultdict(int), 0, 0
    print(f"{'':11}{s}")
    for right, c in enumerate(s):
        count[c] += 1
        dropped = ""
        while count[c] > 1:
            dropped += s[left]
            count[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
        window = " " * left + s[left:right + 1]
        note = f"  (let go of {dropped!r})" if dropped else ""
        print(f"right={right:<2}:  {window:<{len(s)}}  best={best}{note}")


trace_longest("abcabcbb")

**Try it**
- Run `trace_longest("dvdf")` and predict the answer first (3: `"vdf"`). Watch which letters get let go when the second `d` arrives.
- Run `trace_longest("pwwkew")`: at the second `w` the window lets go of `"pw"` in one step. That is the `while` doing its job.
- Add `dict(count)` to the printed line: every letter inside the window has count 1 after the fix, which is the invariant made visible. Letters that already left still show 0; they are harmless here, but they would break a rule that counts distinct keys.

### Where it goes wrong

1. **Recording in the wrong place.** Longest → record *after* the fix. Shortest → record *inside* it. Recording above the `while` on `"abcabcbb"` gives 4.
2. **`if` instead of `while` for the fix.** One step of `left` may not be enough: `"abba"` gives 3 instead of 2, `"abcbd"` gives 4 instead of 3.
3. **Off-by-one length.** The window `[left..right]` has `right - left + 1` items; `right - left` on `"abc"` gives 2.
4. **Order of "let go".** Update the summary with `s[left]` *before* `left += 1`; doing `left += 1` first removes the wrong letter (`"abac"` gives 2 instead of 3).
5. **Distinct counts.** If the rule is "at most k distinct", the number of distinct items is `len(count)`, so delete keys that drop to 0. Without the delete, `exactly_k_distinct([1, 2, 1, 2, 3], 2)` below crashes with `IndexError`.
6. **The jump variant.** Storing `last[c]`, the last index of c, lets `left` jump: `left = max(left, last[c] + 1)`. The `max` matters: without it, `"abba"` gives 3 instead of 2, because the second `a` drags `left` backwards (the cell in Variations shows it).
7. **Negative numbers break it.** Removing an item can make a sum *bigger*, so there is no safe moment to shrink: use prefix sums with a hash map ([Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb#s04)) or the deque follow-up at the end of this section. The map answers an exact sum, the deque a sum of at least k. `shortest_with_sum_at_least([1, -1, 5, 1], 6)` says 4, but `[5, 1]` has length 2.
8. **Degenerate parameters.** `target = 0` makes every window valid, even the empty one, so the shortest loop shrinks past `right` and reads outside the array: `[1]` with target 0 raises `IndexError` without the guard. `k = 0` in "exactly k" calls atMost(−1), and `exactly_k_distinct([1, 2], 0)` raises `IndexError` too. Guard them: `while left <= right and ...`, and `if k < 0: return 0`.

### Edge cases to say out loud

Empty input · one item · all items identical · the whole input is valid · no valid window at all · `k = 0` or `target = 0` · `k >= n` · negatives (method breaks). The asserts below check the cases that apply to the two templates.

In [ ]:
assert longest_without_repeat("") == 0
assert longest_without_repeat("bbbb") == 1
assert longest_without_repeat("abcd") == 4               # whole string valid
assert longest_without_repeat("abba") == 2               # left never moves back
assert shortest_with_sum_at_least([1, 1, 1], 10) == 0    # nothing reaches target
assert shortest_with_sum_at_least([10], 7) == 1
assert shortest_with_sum_at_least([], 1) == 0
assert shortest_with_sum_at_least([1], 0) == 1           # target 0: any single item works
print("edge cases pass")

**Try it**
- Predict, then add: `assert longest_without_repeat("tmmzuxt") == 5` (the window `"mzuxt"`).
- Return `best` without translating `inf` (`return best`): which assert fails first? (`[1, 1, 1], 10` now returns `inf`.)
- Remove `left <= right and` from the shortest template and rerun: the `target = 0` assert crashes with `IndexError`, because every window is valid, even the empty one.

### Variations

Each variation keeps the crawl and changes one thing: the rule, the summary, or what is recorded. The table is the overview, and the cells after it follow its order.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Longest valid** | record after the fix | Longest Substring Without Repeating Characters (3), Longest Repeating Character Replacement (424), Fruit Into Baskets (904), Max Consecutive Ones III (1004) |
| **Shortest valid** | record inside the fix | Minimum Size Subarray Sum (209), Minimum Window Substring (76) |
| **Fixed size k** | no `while`: once `right >= k`, item `right - k` leaves | Permutation in String (567): does `s2` contain an anagram of `s1`; Find All Anagrams in a String (438): every start of one |
| **Budget window** | the rule is "spent ≤ k": zeros flipped, letters replaced | Longest Repeating Character Replacement (424), Max Consecutive Ones III (1004) |
| **Shortest cover** | `formed` counts the letters of `t` that have enough copies | Minimum Window Substring (76) |
| **Count windows, "exactly K"** | exactly(K) = atMost(K) − atMost(K−1); atMost adds `right - left + 1` per step | Subarrays with K Different Integers (992): subarrays with exactly K distinct values; Subarray Product Less Than K (713) |
| **At most k distinct** | `len(count) > k`, with zero-count keys deleted | Fruit Into Baskets (904), where k = 2; Longest Substring with At Most K Distinct Characters (340) |
| **Jump instead of shrink** | `last[c]` lets `left` jump: `left = max(left, last[c] + 1)` | Longest Substring Without Repeating Characters (3) |
| **Window max / min** | the summary is a monotonic deque of indices | Sliding Window Maximum (239): the max of every window of size k; Longest Continuous Subarray With Absolute Diff ≤ Limit (1438): two deques |
| **Take from both ends** | complement: keep a middle window with sum = total − x | Maximum Points You Can Obtain from Cards (1423): k cards from the two ends; Minimum Operations to Reduce X to Zero (1658): remove from the ends until x is used up |
| **Sort first** | sort, then a budget window over the sorted values | Frequency of the Most Frequent Element (1838): the highest count of one value after at most k increments |
| **Cheapest day so far** | degenerate window: `left` jumps to today when today is cheaper | Best Time to Buy and Sell Stock (121): the best profit from one buy and a later sell |
| *Second pass:* **negatives allowed** | the window fails: prefix sums and an increasing deque of starts | Shortest Subarray with Sum at Least K (862): the shortest subarray reaching k, negatives allowed |
| *Second pass:* **window median** | two heaps with lazy deletion | Sliding Window Median (480): the median of every window of size k |
| *Second pass:* **steps of word length L** | L independent windows over word-sized chunks, one per offset | Substring with Concatenation of All Words (30): every start of all the words glued in any order |

**Fixed size.** The simplest change comes first: the window length *is* the rule, so the fix is exactly one item leaving. Permutation in String asks whether `s2` contains a block that is an anagram of `s1`: `"ab"` in `"eidbaooo"` is True, thanks to `"ba"`. Slide a window of `len(s1)` letters over `s2`, keep its letter counts, and compare them with the counts of `s1` after every step.

In [ ]:
def has_permutation(s1, s2):
    k, need, window = len(s1), Counter(s1), Counter()
    for right, c in enumerate(s2):
        window[c] += 1                       # STEP: item enters
        if right >= k:                       # FIX: the item k steps behind leaves
            out = s2[right - k]
            window[out] -= 1
            if window[out] == 0:
                del window[out]              # keep the two Counters comparable
        if window == need:                   # RECORD
            return True
    return False


print(has_permutation("ab", "eidbaooo"), has_permutation("ab", "eidboaoo"))   # True False

**Try it**
- Delete the `if window[out] == 0:` line and the `del` under it: it still prints `True False`, because since Python 3.10 `Counter` equality ignores zero counts. A plain `dict` would not (`{'a': 1, 'b': 0} != {'a': 1}`), so keep the habit.
- Change `if right >= k:` to `if right > k:`. Both calls now print `False`: the first item `s2[0]` never leaves, so the window is one item too big forever. Print `window` to see the stuck `'e'`.
- Try `has_permutation("abc", "ab")`: the window never reaches size 3, so it is `False` without any special case.
- `window == need` costs O(26) per step. Write the O(1) follow-up, a `matches` counter of the letters whose counts agree, and check that it prints `True False` too.

**Budget window with a stale maximum.** A budget is the next step up: the window may hold some bad items, as long as it can pay for them. Longest Repeating Character Replacement asks for the longest stretch that can become one repeated letter after at most k replacements: `"AABABBA"` with k = 1 gives 4. A window qualifies when its length minus the count of its most common letter is at most k. Keeping `max_count` exact while shrinking is expensive, and it never needs to be.

In [ ]:
def character_replacement(s, k):
    count = defaultdict(int)                 # STATE: letter counts in s[left..right]
    left = max_count = best = 0              # max_count: the highest count ever seen
    for right, c in enumerate(s):
        count[c] += 1                        # STEP
        max_count = max(max_count, count[c])
        while (right - left + 1) - max_count > k:    # FIX: too many letters to replace
            count[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)   # RECORD
    return best


print(character_replacement("AABABBA", 1), character_replacement("ABAB", 2))   # 4 4

**Try it**
- Replace `max_count` in the `while` with an exact `max(count.values())`: same answers, just slower. The stale value never lets a *longer* invalid window through, because the answer only grows when some letter reaches a new highest count.
- Change `while` to `if`: still correct here, unlike Longest Substring, because the window grows by at most one per step, so it never needs to shrink by more than one.
- Trace `"AABABBA", 1` by hand, then print `left, right` at each record to check: which window is the first of length 4, and which letter is replaced?

**Shortest window that covers t.** Back to the shortest template, now with a richer summary. Minimum Window Substring asks for the shortest stretch of `s` that contains every letter of `t`, copies included: `"ADOBECODEBANC"` with `t = "ABC"` gives `"BANC"`. Comparing whole counters on every step would cost O(alphabet), so `formed` counts the letters of `t` that already have *enough* copies, and the window is valid when all of them do.

In [ ]:
def min_window(s, t):
    need, window = Counter(t), Counter()     # STATE: what t needs, what the window has
    formed, left = 0, 0                      # formed = letters of t with enough copies
    best_len, best_left = math.inf, 0
    for right, c in enumerate(s):
        window[c] += 1                       # STEP
        if c in need and window[c] == need[c]:
            formed += 1                      # this letter just became satisfied
        while formed == len(need):           # FIX: valid, so tighten it
            if right - left + 1 < best_len:  # RECORD before breaking it
                best_len, best_left = right - left + 1, left
            out = s[left]
            window[out] -= 1
            if out in need and window[out] < need[out]:
                formed -= 1                  # this letter is no longer satisfied
            left += 1
    return "" if best_len == math.inf else s[best_left:best_left + best_len]   # RETURN


print(min_window("ADOBECODEBANC", "ABC"), repr(min_window("a", "aa")))   # BANC ''

**Try it**
- Change `window[c] == need[c]` to `>=` and run `min_window("aab", "ab")`: it returns `'a'` instead of `'ab'`, because the second `a` counted as satisfying a letter again.
- Record the slice instead, `best = s[left:right + 1]` whenever it is shorter: the answers stay the same, but every record now copies the window. Storing `(best_left, best_len)` keeps the record O(1), and it answers "return the substring, not its length" in every template.
- Run `min_window("ab", "b")`: the window opens at index 1. Predict the value of `formed` after each step, then print it.

**Counting windows.** Counting is the third thing a window can record. Subarrays with K Different Integers asks how many subarrays hold exactly K distinct values: `[1, 2, 1, 2, 3]` with K = 2 has 7. "Exactly K" is hard to slide directly, because shrinking cannot tell when to stop, but "at most K" is easy: every window ending at `right` that starts anywhere in `[left..right]` is valid, which is `right - left + 1` windows. So exactly(K) = atMost(K) − atMost(K − 1).

In [ ]:
def at_most_k_distinct(nums, k):
    if k < 0:
        return 0                             # guard: no window has fewer than 0 distinct values
    count, left, windows = defaultdict(int), 0, 0
    for right, x in enumerate(nums):
        count[x] += 1                        # STEP
        while len(count) > k:                # FIX
            count[nums[left]] -= 1
            if count[nums[left]] == 0:
                del count[nums[left]]        # distinct = number of keys
            left += 1
        windows += right - left + 1          # RECORD: all valid windows ending at right
    return windows


def exactly_k_distinct(nums, k):
    return at_most_k_distinct(nums, k) - at_most_k_distinct(nums, k - 1)


print(exactly_k_distinct([1, 2, 1, 2, 3], 2), exactly_k_distinct([1, 2], 0))   # 7 0

**Try it**
- Print both halves: `at_most_k_distinct([1, 2, 1, 2, 3], 2)` is 12 and `at_most_k_distinct([1, 2, 1, 2, 3], 1)` is 5; 12 − 5 = 7. Write the 5 windows with one distinct value by hand.
- Remove the `if count[...] == 0:` line and the `del` under it, then run again: `IndexError`. Zero-count keys still count as distinct, so the `while` can never be satisfied and `left` runs off the end.
- Change `windows += right - left + 1` to `windows += 1`: the answer becomes 0, because you counted one window per end instead of all of them.
- Try `exactly_k_distinct([1, 2, 1, 3, 4], 3)` and predict before running (3).

**The jump variant.** Longest Substring Without Repeating Characters has a faster fix than letting go one letter at a time. Remember the last index of each letter, and when a letter repeats, let `left` jump straight past its old copy. The `max` keeps `left` from moving backwards when that old copy already lies outside the window: `"abba"` gives 2 and `"tmmzuxt"` gives 5.

In [ ]:
def longest_without_repeat_jump(s):
    last, left, best = {}, 0, 0              # STATE: last[c] = most recent index of c
    for right, c in enumerate(s):
        if c in last:
            left = max(left, last[c] + 1)    # FIX: jump past the old copy, never backwards
        last[c] = right                      # STEP
        best = max(best, right - left + 1)   # RECORD
    return best


print(longest_without_repeat_jump("abba"), longest_without_repeat_jump("tmmzuxt"))   # 2 5

**Try it**
- Drop the `max(left, ...)` and write `left = last[c] + 1`: `"abba"` gives 3 and `"tmmzuxt"` gives 6. The old copy was already outside the window, so jumping to it moved `left` backwards.
- Compare with `longest_without_repeat` on `"pwwkew"`: same answer, but here each step is O(1) without an inner loop.
- Print `last` at the end of `"abba"` and predict it first (`{'a': 3, 'b': 2}`).

**Window maximum.** The last summary answers a question no counter can: the maximum of the window after items have left it. Sliding Window Maximum asks for the maximum of every window of size k: `[1, 3, -1, -3, 5, 3, 6, 7]` with k = 3 gives `[3, 3, 5, 5, 6, 7]`. The summary is a deque of indices whose values never increase from front to back, a *monotonic* deque.

A new item kicks out every item behind it that is not bigger: older and not bigger, such an item can never be a maximum again. The front is then the maximum of the window, and it leaves the moment it falls out of the window. Each index is pushed once and popped at most once, so the whole pass is O(n).

In [ ]:
def window_max(nums, k):
    dq, out = deque(), []                    # STATE: indices; nums[dq] never increasing front -> back
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:      # FIX: dominated, older and not bigger
            dq.pop()
        dq.append(i)                         # STEP
        if dq[0] <= i - k:                   # the front fell out of the window
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])          # RECORD: front = max of nums[i-k+1 .. i]
    return out


print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))   # [3, 3, 5, 5, 6, 7]

**Try it**
- Change `<=` to `<` in the pop condition and run `window_max([5, 5, 5, 5], 4)` with `print(len(dq))` right after `dq.append(i)`: the answers stay right, but equal values pile up, and the length reaches 4 instead of staying 1.
- Delete the two lines that expire the front and run `window_max([5, 1, 1, 1], 2)`: you get `[5, 5, 5]` instead of `[5, 1, 1]`, because the 5 is reported long after it left.
- Store values instead of indices: append `x`, pop while `dq[-1] <= x`, and expire with `if i >= k and dq[0] == nums[i - k]: dq.popleft()`. `window_max([3, 3, 1], 2)` gives `[3, 1]` instead of `[3, 3]`, because the expiry removed the newer 3. A values deque needs `<` in the pop to keep duplicates; with indices, expiry is one comparison.
- With `k = 1` the output is the input itself: run `window_max([4, 2, 7], 1)` to confirm.

Three mediums reuse these windows with one twist each. Maximum Points You Can Obtain from Cards takes k cards from the two ends, and Minimum Operations to Reduce X to Zero removes items from the two ends until they sum to x. Both flip into a middle window, the items left behind: the n − k cards with the smallest sum, or the longest stretch whose sum is total − x.

Frequency of the Most Frequent Element sorts first, because after sorting the cheapest values to raise to `nums[right]` are its nearest left neighbours. A window can be raised to its last value for `nums[right] * length - window_sum` increments, so it is broken when that cost passes k, and the longest unbroken window is the answer.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

**Negatives allowed.** Shortest Subarray with Sum at Least K is Minimum Size Subarray Sum with negative numbers allowed: `[2, -1, 2]` with k = 3 gives 3, the whole array. The window fails, so work on prefix sums `P`, where the subarray `(i, j]` has sum `P[j] - P[i]`. For each end `j`, keep the useful starts in a deque with increasing `P`. A start whose `P` is not smaller than a later start's is useless, because the later one is shorter and at least as good.

In [ ]:
def shortest_sum_at_least_k(nums, k):
    prefix = [0] + list(accumulate(nums))
    dq, best = deque(), math.inf             # STATE: indices into prefix, prefix values increasing
    for j, p in enumerate(prefix):
        while dq and p - prefix[dq[0]] >= k: # this start is done: j is its best end
            best = min(best, j - dq.popleft())   # RECORD
        while dq and prefix[dq[-1]] >= p:    # FIX: a later, lower start is strictly better
            dq.pop()
        dq.append(j)                         # STEP
    return -1 if best == math.inf else best  # RETURN


print(shortest_sum_at_least_k([2, -1, 2], 3), shortest_sum_at_least_k([1, 2], 4))   # 3 -1

**Try it**
- Print `prefix` for `[2, -1, 2]`: `[0, 2, 1, 3]`. Every subarray sum is a difference of two of these numbers.
- Delete the second `while` loop and run `shortest_sum_at_least_k([-5, 6], 6)`: it returns -1 instead of 1. The useless start 0 (prefix 0, higher than the later prefix −5) blocks the front of the deque.
- Run `shortest_sum_at_least_k([1, -1, 5, 1], 6)`: 2, the case where the simple template said 4.

Sliding Window Median asks for the median of every window of size k: `[1, 3, -1, -3, 5, 3, 6, 7]` with k = 3 gives `[1, -1, -1, 3, 5, 6]`. Keep the window in the two heaps of the median finder from [Heaps](07_Heaps_Intervals_Greedy.ipynb#s13), a max-heap for the lower half and a min-heap for the upper half. A heap cannot delete from its middle, so a leaving item is only noted in a dict of pending deletions and popped once it reaches a top: lazy deletion. Balance the halves by their live counts, never by their lengths.

Substring with Concatenation of All Words asks for every start of a block made of all the given words, each used once, in any order; the words share one length L. `"barfoothefoobarman"` with `["foo", "bar"]` gives `[0, 9]`. Cut `s` into chunks of length L, once for each offset 0..L−1, and slide a counting window over the chunks, as in Permutation in String. A chunk that is not a word empties the window, and a word seen too often shrinks it from the left: O(n · L) in total.

### Say it in the interview

> "The brute force tries every start and rescans to the right: O(n²). Neighbouring windows share all but their end items, so I'll keep a window with a count map that I update in O(1) as items enter and leave. `right` adds one item per step; while the window breaks the rule I move `left`. `left` never has to come back: once a window is broken, every longer window containing it is broken too. Both pointers only move forward, so it's O(n) time and O(alphabet) space."

Then name the invariant out loud ("after this loop the window has no repeats") and point at the line where you record. Likely follow-ups and your answers:

- *Return the substring, not the length* → store `best_left` when you record.
- *Numbers can be negative* → the rule is not monotone; prefix sums with a hash map, or the deque follow-up.
- *The input is a stream* → the window works online: it only ever needs the current window's summary.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Best Time to Buy and Sell Stock | `sliding_window/best_time_to_buy_and_sell_stock.py` | degenerate window: keep the cheapest price so far; profit = today − cheapest (coded in [From Idea to Code](01_Start_Here.ipynb#s01)) |
| Fruit Into Baskets | `sliding_window/fruit_into_baskets.py` | longest window with ≤ 2 distinct values; distinct = number of dict keys |
| Longest Repeating Character Replacement | `sliding_window/longest_repeating_character_replacement.py` · `practice/simple/12_longest_repeating_character_replacement.py` | valid iff length − max_count ≤ k; a stale max_count is safe because only longer windows can improve the answer |
| Longest Substring Without Repeating Characters | `sliding_window/longest_substring_without_repeating_characters.py` · `practice/simple/09_longest_substring_without_repeating_characters.py` | shrink while the new letter is doubled, or jump `left = max(left, last[c] + 1)` |
| Max Consecutive Ones III | `sliding_window/max_consecutive_ones_iii.py` | budget window: at most k zeros inside |
| Minimum Size Subarray Sum | `sliding_window/minimum_size_subarray_sum.py` | non-negative numbers: tighten while sum ≥ target, record inside the loop |
| Minimum Window Substring | `sliding_window/minimum_window_substring.py` · `practice/simple/10_minimum_window_substring.py` | `formed` counts letters with enough copies; tighten while all are satisfied |
| Permutation in String | `sliding_window/permutation_in_string.py` | fixed window of len(s1); compare letter counts |
| Shortest Subarray with Sum at Least K | `sliding_window/shortest_subarray_with_sum_at_least_k.py` | negatives break the window: prefix sums + increasing deque of starts |
| Sliding Window Maximum | `sliding_window/sliding_window_maximum.py` · `practice/simple/11_sliding_window_maximum.py` | deque of indices with never-increasing values; pop dominated items, front is the max |
| Sliding Window Median | `sliding_window/sliding_window_median.py` | two heaps (low max-heap, high min-heap) with lazy deletion of leaving items |
| Subarrays with K Different Integers | `sliding_window/subarrays_with_k_different_integers.py` | exactly K = atMost(K) − atMost(K−1); atMost adds `right - left + 1` per step |
| Substring with Concatenation of All Words | `sliding_window/substring_with_concatenation_of_all_words.py` | move in word-sized steps; one window per offset 0..L−1 |

### Self-check

1. Why does "record" sit inside the fix loop for *shortest* problems but after it for *longest* ones?
<details><summary>Answer</summary>For shortest problems, each <code>left</code> is recorded with the first <code>right</code> that makes it valid, which is its shortest window; longer windows with the same start are useless, so <code>left</code> moves on. For longest problems the window is only guaranteed valid once the loop has finished, and at that moment it is the longest valid window ending at <code>right</code>.</details>

2. Longest subarray with sum ≤ k, numbers may be negative: does the template work, and what would you do?
<details><summary>Answer</summary>No: removing a negative number from the left makes the sum larger, so the rule is not monotone. For each end j you need the earliest i with P[i] ≥ P[j] − k. Only positions where the prefix sum reaches a new maximum can be that earliest i, so keep them, in increasing order of P, and binary-search them with bisect: O(n log n).</details>

3. In Longest Repeating Character Replacement, why is it fine to never decrease `max_count` while shrinking?
<details><summary>Answer</summary>The answer only grows when a window has a larger max_count than ever before. A stale, too-large max_count can only keep the window at a length already achieved; it never produces a longer, invalid answer.</details>

<a id="s20"></a>

## Strings

> A Python string is a frozen row of characters. Reading `s[i]` costs O(1), but every "change" builds a brand-new string. So string algorithms are about **reading** cleverly, with one cursor `i` that only moves right or two pointers closing in from both ends, plus a little memory of what has been seen; and about **building** the answer in a list that you `"".join` once at the end.

[Sliding Window](#s06) moved two fingers to the right and kept a summary between them. A string parser needs only one finger: a cursor that never moves back. This section is that cursor and the toolbox around it: splitting, reading numbers, keys, palindromes, runs and carries.

**Reach for it when** the input is text and you must **parse** it: split it, read a number or a version. Or **reshape** it: reverse the words, compress runs, write a zigzag. Or compare **letters**, as in shifted strings and palindromes; do **arithmetic on digits** given as text; or find a **repeat**, such as a prefix that is also a suffix or the longest repeated substring. "Longest substring such that ..." is usually a [Sliding Window](#s06) instead.

**In this repo:** `strings/` (17 problems) · bank: `practice/simple/50_longest_palindromic_substring.py` · basics: `practice/simple/basics/strings/01_character_counting_and_anagrams.py`, `practice/simple/basics/strings/02_two_pointer_palindromes_and_reverse_words.py`, `practice/simple/basics/strings/03_kmp_prefix_function.py`, `practice/simple/basics/strings/04_rabin_karp_rolling_hash.py`, `practice/simple/basics/strings/05_encode_decode_strings_and_join.py`

### The picture

Reading: one cursor that never steps back. Here it splits a string by hand, the classic warm-up.

```text
s = "ab,,c,"   sep = ","      i = the cursor, it only moves right;  start = where the open piece began

    a b , , c ,            index 0 1 2 3 4 5, and the end is 6
   [a b]                   i=2 is ",":  emit s[0:2] = "ab"   start = 3
         []                i=3 is ",":  emit s[3:3] = ""     start = 4    two commas in a row
           [c]             i=5 is ",":  emit s[4:5] = "c"    start = 6
               []          the end:     emit s[6:6] = ""                 the last piece has no "," after it

=> ["ab", "", "c", ""]     the same as "ab,,c,".split(",")
```

Building: collect the pieces in a list and `"".join` them once. Each `out += piece` builds a new string and may copy everything built so far, O(n²) characters over n appends in the worst case. CPython sometimes grows a string in place, but that is an implementation detail, not a promise.

Why it is fast: the brute-force split calls `find`, then slices off the remainder for every piece, which copies the rest of the string each time: O(n²) characters at worst. The cursor keeps one index, `start`, instead of a copy, so each position is compared with the separator once: O(n·m) for a separator of length m, O(n) for one character.

### From idea to code

**The idea in one sentence:** *walk a cursor `i` from left to right; each character either extends the open token or ends it; when a token ends, emit it and open the next one, and emit the last one after the loop.*

The **State** is the cursor `i` plus the **open token**, kept as where it began, `start`, or as its running value, `num` and `sign`; and the output list. The **Definition**: `s[start:i]` is the token read so far, and everything before `start` has been emitted. The **Invariant**, true after every step: every separator that starts before `i` has been cut, and `i` never moves left. A **Step** looks at `s[i]` and either extends the token, `i += 1`, or ends it, emits it and opens the next one past the separator.

The **Record** happens when a token ends, **and once more after the loop**, because the last token has no separator after it; a number parser records early instead, clamping the moment it overflows. **Init** is `i = start = 0` with `parts = []`, or `num, sign = 0, 1`. The **Return** is `parts`, `sign * num` or a flag, and you say out loud what empty input gives: `[""]` for a split, 0 for a number.

The warm-up you are most likely to meet is Implement String Split: cut `"ab,,c,"` at every comma into `['ab', '', 'c', '']`, exactly as `str.split(",")` does. Its sibling is `s.split()` with no argument, where a run of spaces counts as one gap and no empty word comes back. The piece so far is `s[start:i]`, and skipping a gap is `while i < n and s[i] == " ": i += 1`. In both functions the order of two lines is a decision: close the piece with the *old* `start` (RECORD), then move `start` (STEP).

In [ ]:
def split_on_char(s, sep):                   # the 6-line answer for a one-character sep
    parts, start = [], 0                     # STATE + INIT: s[start:i] is the open piece
    for i, ch in enumerate(s):
        if ch == sep:
            parts.append(s[start:i])         # RECORD: close the piece (it may be "")
            start = i + 1                    # STEP: the next piece opens after the separator
    parts.append(s[start:])                  # RECORD the last piece: no separator follows it
    return parts                             # RETURN


def split_words(s):                          # like s.split(): runs of spaces, no empty words
    words, i, n = [], 0, len(s)              # STATE + INIT: i = the cursor
    while i < n:
        while i < n and s[i] == " ":         # phase 1: skip the gap
            i += 1
        start = i                            # STEP: a word opens here
        while i < n and s[i] != " ":         # phase 2: read the word
            i += 1
        if start < i:                        # a trailing gap reads an empty word: drop it
            words.append(s[start:i])         # RECORD
    return words                             # RETURN


print(split_on_char("a,,b,", ","), split_on_char("", ","))   # ['a', '', 'b', ''] ['']
print(split_words("  the sky   is blue "))                   # ['the', 'sky', 'is', 'blue']

**Try it**
- Delete the last `parts.append(s[start:])` and run `split_on_char("a,b", ",")`: `['a']`. The final piece never meets a separator, so only that line can emit it.
- Change `start = i + 1` to `start = i`: `split_on_char("a,b", ",")` gives `['a', ',b']`, because the separator leaks into the next piece.
- Swap the two lines inside the `if` (move `start` first) and run `split_on_char("a,b", ",")`: `['', 'b']`. Every closed piece is now `s[i + 1:i]`, which is empty.
- Drop the `if start < i:` test (append every time) and run `split_words("the sky  ")`: `['the', 'sky', '']`. A trailing gap reads an empty word.

A separator of any length needs one more idea. A match can start at `i` only while `i <= len(s) - m`, it is tested with `s[i:i + m] == sep`, which is what `s.startswith(sep, i)` does, and the cursor then jumps past the *whole* match. Matches never overlap, so `"aaa"` split on `"aa"` gives `['', 'a']`, as in Python.

Reading a number adds phases, the shape of every number parser. String to Integer (8), the classic atoi, reads an integer from the front of a text: skip the spaces, take at most one sign, read the digits, and clamp to the 32-bit range, so `"   -042abc"` gives −42. A digit is `"0" <= c <= "9"`, its value is `ord(c) - ord("0")`, and it joins the number as `num = num * 10 + d`. The cell holds both parsers.

In [ ]:
def split_by_hand(s, sep):                   # any non-empty sep: the same result as s.split(sep)
    if not sep:
        raise ValueError("empty separator")  # what Python does; ask what they want
    parts, start, i, m = [], 0, 0, len(sep)  # STATE + INIT: s[start:i] is the open piece
    while i <= len(s) - m:                   # could a separator start at i?
        if s[i:i + m] == sep:
            parts.append(s[start:i])         # RECORD: close the piece (it may be "")
            i += m                           # STEP: jump over the whole separator ...
            start = i                        # ... and open the next piece right after it
        else:
            i += 1                           # STEP: the open piece grows by one character
    parts.append(s[start:])                  # RECORD the last piece
    return parts                             # RETURN


def my_atoi(s):
    INT_MIN, INT_MAX = -2**31, 2**31 - 1
    i, n, sign, num = 0, len(s), 1, 0        # STATE + INIT: cursor, sign, magnitude so far
    while i < n and s[i] == " ":             # phase 1: leading spaces
        i += 1
    if i < n and s[i] in "+-":               # phase 2: at most one sign
        sign = -1 if s[i] == "-" else 1
        i += 1
    while i < n and "0" <= s[i] <= "9":      # phase 3: digits
        num = num * 10 + (ord(s[i]) - ord("0"))   # STEP: append the digit
        if sign * num > INT_MAX:             # RECORD early: clamp the moment it overflows
            return INT_MAX
        if sign * num < INT_MIN:
            return INT_MIN
        i += 1
    return sign * num                        # RETURN: whatever follows the digits is ignored


print(split_by_hand("a,,b,", ","), split_by_hand("aaa", "aa"))       # ['a', '', 'b', ''] ['', 'a']
print(my_atoi("   -042abc"), my_atoi("+7"), my_atoi("-91283472332"))  # -42 7 -2147483648

**Try it**
- Change `<=` to `<` in `while i <= len(s) - m:` and run `split_by_hand("a,", ",")` and `split_by_hand(",", ",")`: `['a,']` and `[',']` instead of `['a', '']` and `['', '']`, because the last position is never tested. Then try `while i < len(s):`: every result is right again.
- Change `i += m` to `i += 1` and run `split_by_hand("aaa", "aa")`: `['', '', 'a']` instead of `['', 'a']`, because the two matches overlap.
- Add `print(i, s[i], num)` right after the `num = ...` line and run `my_atoi("  -042x")`: `3 0 0`, `4 4 4`, `5 2 42`. The number is built without its sign; the sign is applied at the end.

### Watch it work

The trace runs the general split on `"ab,,c,"` and prints a line each time a piece is closed: where the cursor found a separator, which slice it emitted, and where the next piece starts. The last line is the tail, the one piece the loop itself never closes.

In [ ]:
def trace_split(s, sep):
    parts, start, i, m = [], 0, 0, len(sep)
    print(f"s={s!r}  sep={sep!r}")
    while i <= len(s) - m:
        if s[i:i + m] == sep:
            parts.append(s[start:i])
            print(f"i={i}: separator -> emit s[{start}:{i}] = {s[start:i]!r:5} next piece starts at {i + m}")
            i += m
            start = i
        else:
            i += 1
    parts.append(s[start:])
    print(f"end: emit the last piece s[{start}:] = {s[start:]!r}")
    return parts


print(trace_split("ab,,c,", ","))   # ['ab', '', 'c', '']

**Try it**
- Run `trace_split(",a,,", ",")` and predict the pieces first: `['', 'a', '', '']`. A separator at the very start closes an empty first piece.
- Run `trace_split("a--b---c", "--")`: `['a', 'b', '-c']`. After the match at index 4 the cursor jumps to 6, so the third `-` stays in the last piece, exactly as `str.split` does.
- Swap `i += m` and `start = i` (set `start` first) and run `trace_split("a,b", ",")`: `['a', ',b']`. The new piece now starts *on* the separator.
- Give `split_by_hand` a parameter `maxsplit=-1` and loop `while i <= len(s) - m and len(parts) != maxsplit:`. It now matches `s.split(sep, maxsplit)`: `split_by_hand("a,b,c", ",", 1)` is `['a', 'b,c']`, and the tail append needs no change.

### Where it goes wrong

1. **Forgetting the last piece.** Emitting only when a separator is seen drops the tail: `"a,b"` gives `["a"]`. Fix: `parts.append(s[start:])` after the loop; it also makes `""` give `[""]` and `"a,"` give `["a", ""]`, like Python.
2. **`+=` in a loop.** Each `+=` builds a new string and may copy everything so far: O(n²) characters over n appends. Fix: append pieces to a list, `"".join` once.
3. **Reading `s[i]` before checking `i < n`.** `"-"` or `"   "` makes a parser read past the end (`IndexError`). Fix: every `while` starts with `i < n and ...`.
4. **A branch that never moves the cursor.** Every path through the body of `while i < n` must advance `i` or leave the loop; a branch that forgets spins forever on the first character that reaches it. Replace the `i += 1` of phase 2 in `split_words` with `pass`, and `split_words("a")` never returns. Fix: before running, point at the `i += ...` in each branch.
5. **`isdigit()` as the digit test.** `"²".isdigit()` is True, yet `int("²")` raises `ValueError`. Fix: `"0" <= c <= "9"`.
6. **Clamping too late, or not at all.** Python ints never overflow, so the bug hides: `"-91283472332"` must give `-2147483648`. Fix: check after every digit; in Java or C++, check *before* multiplying, with `num > (INT_MAX - d) / 10`.
7. **Moving by 1 after a separator match.** Matches must not overlap: `"aaa"` split on `"aa"` is `["", "a"]`. Fix: `i += len(sep)`.
8. **Comparing numbers as strings.** `"1.01" != "1.001"` as strings, yet they are the same version, and `"10" < "9"` is True. Fix: parse each field into an int (`x = x * 10 + d`).
9. **A 26-slot count on text that isn't lowercase.** `ord(ch) - ord("a")` sends `"Z"` into the `'t'` slot and `"A"` out of range ([Arrays & Hashing](02_Arrays_Hashing_Prefix_Sums.ipynb#s03), trap 3). Ask the alphabet; otherwise key on `"".join(sorted(w))` or a `Counter`.
10. **Deleting while scanning.** Removing characters from the list you are indexing shifts every later index: `del chars[i]` in Minimum Remove turns `"a)b)c"` into `'ab)'`. Fix: mark (`chars[i] = ""`) during the scan and join once at the end.
11. **The palindrome expansion overshoots.** The `while` stops one step past the palindrome on both sides, `lo = -1` and `hi = 3` on `"aba"`, so the palindrome is `s[lo + 1:hi]`, of length `hi - lo - 1`.

### Edge cases to say out loud

Empty string · only separators, only spaces · a separator at the start, at the end, twice in a row · a separator that overlaps itself (`"aaa"` on `"aa"`) · one character · a sign with no digits (`"+"`) · overflow both ways · leading zeros · uppercase, digits or spaces where the code assumes lowercase letters (ask the alphabet) · non-ASCII text (Python indexes code points; one visible character can be several of them). The asserts below check the splitters against Python's own `split` and walk `my_atoi` through the number cases.

In [ ]:
for text in ["", ",", ",a,", "abc", "a,,b,"]:
    assert split_on_char(text, ",") == split_by_hand(text, ",") == text.split(",")
for text in ["", "   ", " a  b ", "x"]:
    assert split_words(text) == text.split()
assert split_by_hand("aaa", "aa") == ["", "a"] and split_by_hand("a--b----c", "--") == ["a", "b", "", "c"]
assert my_atoi("") == 0 and my_atoi("   ") == 0      # nothing to read
assert my_atoi("+") == 0 and my_atoi("-") == 0       # a sign with no digits
assert my_atoi("+-12") == 0                          # only one sign is allowed
assert my_atoi("  + 4") == 0                         # no space between sign and digits
assert my_atoi("2147483647") == 2**31 - 1            # exactly INT_MAX is fine
assert my_atoi("2147483648") == 2**31 - 1            # one more: clamp
assert my_atoi("-2147483648") == -2**31              # INT_MIN itself is reachable
assert my_atoi("0000123") == 123                     # leading zeros
assert my_atoi("²") == 0                             # not an ASCII digit
print("edge cases pass")

**Try it**
- Compare `" a  b ".split(" ")` with `" a  b ".split()`: `['', 'a', '', 'b', '']` versus `['a', 'b']`. The first is `split_on_char`, the second is `split_words`; ask which one the problem means.
- Predict, then check: `my_atoi("-0012a42")` is -12 and `my_atoi("3.14")` is 3; the scan stops at the first non-digit.
- Add `assert split_on_char("a", "a") == ["", ""]` and predict whether it passes before running it (it does: one separator, two empty pieces).

### Variations

Every variation is the same cursor with one more piece of memory: a second cursor, a flag, a key, a center, a carry. The table is the overview; its first three rows are the templates above, and the rest follow the order of the cells below.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Split on one character** | emit at each separator, and once more at the end | Implement String Split, the warm-up |
| **Split on runs of spaces** | phases: skip the gap, read a word; drop the empty word a trailing gap reads | Reverse Words in a String (151): the words in reverse order; Length of Last Word (58) |
| **Read a number** | phases spaces → sign → digits; `num = num * 10 + d`; clamp | String to Integer (8) |
| **Two cursors in lockstep** | read one field from each string, compare, step over the dots | Compare Version Numbers (165): which of two versions is larger |
| **Quoted fields** | a flag flips at each `"`; a separator cuts only outside quotes | the split follow-up |
| **Difference key** | the tuple of `(s[i+1] - s[i]) % 26` forgets the shift | Group Shifted Strings (249): group the words that shift into each other |
| **Expand around center** | 2n − 1 centers; grow while both ends match | Longest Palindromic Substring (5); Palindromic Substrings (647): count them all |
| **One deletion allowed** | two pointers; at the first mismatch, try skipping either side | Valid Palindrome II (680): a palindrome after deleting at most one letter |
| **Runs** | `j` walks to the end of the run; the next run starts at `j` | Expressive Words (809): can a word stretch into `s`; String Compression (443): `aaabb` → `a3b2` in place; Count and Say (38): read the previous term's runs aloud |
| **Digits with a carry** | walk both strings from the right; `divmod(d, 10)`; build backwards | Add Strings (415): add two numbers given as text; Add Binary (67); Multiply Strings (43) |
| **Mark, then join** | stack of open-bracket indices; blank what can't be matched | Minimum Remove to Make Valid Parentheses (1249) |
| **Reverse twice** | reverse the whole list, then reverse each word back | Reverse Words in a String (151); Reverse Words in a String II (186): the same, in place |
| **Row buckets** | a row index bouncing between 0 and numRows − 1; one list per row | Zigzag Conversion (6): write the string in a zigzag, read it row by row |
| *Second pass:* **grammar flags** | three flags instead of a value; the `e` resets "digit seen" | Valid Number (65): is the text a decimal or scientific number |
| *Second pass:* **chunks of three** | base 1000: each chunk is spelled alike, then its scale word | Integer to English Words (273): spell a number in English words |
| *Second pass:* **cut + hash map** | cut each word in two; one half a palindrome, the other half's reverse a whole word | Palindrome Pairs (336): the pairs of words that join into a palindrome |
| *Second pass:* **KMP borders** | `fail[i]` = the longest proper border of `s[:i + 1]`; on a mismatch, fall back | Longest Happy Prefix (1392): the longest prefix that is also a suffix; Shortest Palindrome (214): the fewest letters added in front; Find the Index of the First Occurrence in a String (28) |
| *Second pass:* **rolling hash + binary search** | binary search the length; a rolling hash tests one length in O(n) | Longest Duplicate Substring (1044): the longest substring that occurs twice |
| *Second pass:* **greedy packing + divmod** | pack words greedily; `divmod(spaces, gaps)` puts the extras on the left | Text Justification (68): lines of exactly `width` characters |

Many string problems are taught where their technique lives:

| Problems | Technique | Section |
|---|---|---|
| longest without repeats, minimum window, permutation in string, all anagrams (3, 76, 567, 438) | a window with letter counts | [Sliding Window](#s06) |
| valid palindrome with punctuation (125) | two pointers that skip non-letters | [Two Pointers](#s05) |
| brackets, decode string, calculators, simplify path (20, 394, 224, 227, 71) | a stack | [Stacks & Queues](04_Stacks_Queues_Linked_Lists.ipynb#s07) |
| anagram counts, group anagrams, encode/decode strings (242, 49, 271) | a fingerprint key; length-prefix framing | [Arrays & Hashing](02_Arrays_Hashing_Prefix_Sums.ipynb#s03) |
| word search II, word squares (212, 425) | a trie walked along the board | [Tries](06_Trees_Tries.ipynb#s12) |
| phone letters, IP addresses, palindrome partitioning (17, 93, 131) | backtracking over cuts | [Backtracking](08_Backtracking.ipynb#s16) |

**Two cursors and a quote flag.** The first variations stay with parsing. Compare Version Numbers compares two versions field by field as integers and returns −1, 0 or 1: `"1.01"` equals `"1.001"`, and `"1.0"` is smaller than `"1.0.1"`. Run two cursors in lockstep, read one field from each string as a number, and let the first different field decide. A string that has run out simply reads as 0, which is exactly the rule that missing revisions count as 0.

In [ ]:
def compare_version(v1, v2):                 # O(n + m) time, O(1) extra space
    i = j = 0
    while i < len(v1) or j < len(v2):        # until BOTH strings are used up
        x = 0
        while i < len(v1) and v1[i] != ".":  # read one field; a used-up string gives 0
            x = x * 10 + int(v1[i])
            i += 1
        y = 0
        while j < len(v2) and v2[j] != ".":
            y = y * 10 + int(v2[j])
            j += 1
        if x != y:
            return -1 if x < y else 1        # the first different field decides
        i += 1                               # step over the dots
        j += 1
    return 0


print(compare_version("1.01", "1.001"), compare_version("1.0", "1.0.1"), compare_version("1.10", "1.9"))   # 0 -1 1

**Try it**
- Change `or` to `and` in `while i < len(v1) or j < len(v2)` and run `compare_version("1.0.1", "1")`: 0 instead of 1. The loop stopped when the shorter string ran out.
- Run `"10" < "9"` and `10 < 9`: `True` and `False`. Strings compare character by character, which is why each field is parsed into an int before `compare_version("1.10", "1.9")` can return 1.
- Run `compare_version("1", "1.0.0.0")`: 0. The used-up string keeps reading 0 while the other one still has fields.

Quoted fields are the classic follow-up to the split warm-up: `'1,"Smith, John",42'` must give `['1', 'Smith, John', '42']`. One flag remembers whether the cursor is inside quotes. A quote flips the flag and is not kept, a separator closes the field only while the flag is off, and every other character joins the open field. The last field is emitted after the loop, exactly as in the warm-up.

**Keys: group by "the same up to ...".** Group Shifted Strings groups the words that turn into each other when every letter moves the same number of steps along the alphabet: `"abc"`, `"bcd"` and `"xyz"` form one group, `"az"` and `"ba"` another. Compute a key that forgets exactly that change, and a dict does the grouping. A shift forgets the starting letter but keeps the gaps between neighbours, taken mod 26 so that `z → a` is a gap of 1. Anagrams forget the order instead; their key is built in [Arrays & Hashing](02_Arrays_Hashing_Prefix_Sums.ipynb#s03).

In [ ]:
def group_by_key(words, key):
    groups = defaultdict(list)               # key -> the words that share it
    for w in words:
        groups[key(w)].append(w)
    return list(groups.values())


def shift_key(w):                            # the gaps between neighbours, mod 26
    return tuple((ord(b) - ord(a)) % 26 for a, b in zip(w, w[1:]))


print(group_by_key(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"], shift_key))
# [['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]

**Try it**
- Drop the `% 26` from `shift_key` and rerun: `'az'` and `'ba'` land in different groups, because their gaps are now 25 and -1.
- Print `shift_key("a")` and `shift_key("z")`: both `()`, so all one-letter words form one group. That is right: any letter shifts into any other.
- Predict which group `"cb"` joins before adding it: `shift_key("cb")` is `(25,)`, the same as `"az"` and `"ba"`, so all three shift into each other.

**Palindromes: from the middle, and from the ends.** Longest Palindromic Substring asks for the longest block that reads the same both ways: `"babad"` gives `"bab"`, and `"cbbd"` gives `"bb"`. A palindrome is decided by its center, and growing it one step on each side costs one comparison. So try all 2n − 1 centers, each letter for odd lengths and each gap for even lengths, and stop at the first mismatch, since no wider palindrome can share that center.

Valid Palindrome II asks whether deleting at most one letter makes the string a palindrome: `"abca"` becomes one without its `b` or its `c`, and `"abc"` never does. It walks two pointers inward instead. At the first mismatch one of the two letters must go, so check both leftovers; with punctuation to skip, the walk is in [Two Pointers](#s05).

In [ ]:
def longest_palindrome(s):                   # O(n²) time, O(1) extra space
    best_lo, best_len = 0, 0
    for center in range(len(s)):
        for lo, hi in ((center, center), (center, center + 1)):   # odd and even centers
            while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
                lo -= 1                      # grow while both ends match
                hi += 1
            if hi - lo - 1 > best_len:       # the loop overshot by one on each side
                best_lo, best_len = lo + 1, hi - lo - 1
    return s[best_lo:best_lo + best_len]


def valid_palindrome_ii(s):                  # a palindrome after deleting at most one letter? O(n)
    def is_pal(lo, hi):
        while lo < hi:
            if s[lo] != s[hi]:
                return False
            lo, hi = lo + 1, hi - 1
        return True
    lo, hi = 0, len(s) - 1
    while lo < hi:
        if s[lo] != s[hi]:                   # the first mismatch: one of these two must go
            return is_pal(lo + 1, hi) or is_pal(lo, hi - 1)
        lo, hi = lo + 1, hi - 1
    return True


print(longest_palindrome("babad"), longest_palindrome("cbbd"))   # bab bb
print(valid_palindrome_ii("abca"), valid_palindrome_ii("abc"))   # True False

**Try it**
- Keep only the odd center, `for lo, hi in ((center, center),):`, and run `longest_palindrome("cbbd")`: `'c'`. The palindrome `"bb"` has no middle letter, so only a gap can be its center.
- Add `print(center, lo, hi)` after the `while` loop and run `longest_palindrome("aba")`: the line `1 -1 3` shows the loop stopping one step outside `s[0:3]` on both sides.
- Keep only the left skip (`return is_pal(lo + 1, hi)`) and run `valid_palindrome_ii("cbbcc")`: False instead of True. Deleting the `c` at index 3 works; deleting the `b` at index 1 does not.

**Runs and carries.** A *run* is a block of equal letters: a second cursor `j` walks to its end, and the next run starts at `j`. Expressive Words asks whether a word can be stretched into `s` by growing some of its runs to length 3 or more: `"hello"` stretches into `"heeellooo"`, and `"helo"` does not. Compare the runs of the two strings pair by pair; String Compression and Count and Say write the runs out instead.

Add Strings adds two non-negative numbers given as text: `"999"` plus `"1"` is `"1000"`. Walk both strings from the right with a carry, as on paper, and build the answer backwards; a string that has run out adds nothing, and a last carry needs a turn of its own. All of these are O(n).

In [ ]:
def runs(s):                                 # "aaabcc" -> [('a', 3), ('b', 1), ('c', 2)]
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:   # j walks to the end of this run
            j += 1
        out.append((s[i], j - i))            # RECORD the run
        i = j                                # STEP: the next run starts where this one ended
    return out


def expressive(s, word):                     # can word be stretched into s? (809)
    a, b = runs(s), runs(word)
    return len(a) == len(b) and all(c1 == c2 and (k1 == k2 or (k1 >= 3 and k1 > k2))
                                    for (c1, k1), (c2, k2) in zip(a, b))


def add_strings(a, b):                       # 415: digit by digit from the right
    i, j, carry, out = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:         # any digit or carry left
        d = carry
        if i >= 0:
            d += ord(a[i]) - ord("0")
            i -= 1
        if j >= 0:
            d += ord(b[j]) - ord("0")
            j -= 1
        carry, digit = divmod(d, 10)
        out.append(str(digit))
    return "".join(reversed(out))            # built backwards: reverse once


print(runs("aaabcc"), expressive("heeellooo", "hello"), expressive("heeellooo", "helo"))
# [('a', 3), ('b', 1), ('c', 2)] True False
print(add_strings("999", "1"), add_strings("11", "123"))   # 1000 134

**Try it**
- Change `i = j` to `i = j + 1` and run `runs("aaabcc")`: `[('a', 3), ('c', 2)]`. The `b` run vanishes, because the next run already starts at `j`.
- Predict `expressive("zzzzzyyyyy", "zzyy")` and `expressive("aaa", "aaaa")`: True (both runs can grow to 5) and False (a run can't shrink).
- Drop `or carry` from the loop and run `add_strings("999", "1")`: `'000'`. The last carry needs one more turn of the loop.

**Mark, then join; reverse twice; row buckets.** The last three reshape a string in one pass each. Minimum Remove to Make Valid Parentheses deletes the fewest brackets so that the rest is balanced: `"lee(t(c)o)de)"` loses its last `)`. A stack of the indices of `(` still open finds exactly what must go: a `)` with nothing to close, and every `(` still open at the end. Blank them during the scan, and join once.

Reverse Words in a String puts the words in reverse order with single spaces: `"  the sky   is blue "` becomes `"blue is sky the"`. In Python, say `" ".join(reversed(s.split()))` first; reversing twice is the follow-up for a mutable list of characters. Reversing the whole list puts the words in the right order but spells each one backwards, and a second sweep reverses each word back.

Zigzag Conversion writes the string down and up across `num_rows` rows and reads it row by row: `"PAYPALISHIRING"` on 3 rows reads `"PAHNAPLSIIGYIR"`. Only the *row* of each character matters, and the row bounces 0, 1, ..., num_rows − 1, ..., 1, 0, so keep one list per row and a step of +1 or −1.

In [ ]:
def min_remove_to_make_valid(s):
    chars, open_idx = list(s), []            # open_idx = indices of "(" still waiting for a partner
    for i, ch in enumerate(s):
        if ch == "(":
            open_idx.append(i)
        elif ch == ")":
            if open_idx:
                open_idx.pop()               # matched with the most recent "("
            else:
                chars[i] = ""                # nothing to close: mark it, delete later
    for i in open_idx:
        chars[i] = ""                        # never closed
    return "".join(chars)                    # every deletion happens here, at once


def reverse_words(s):
    chars = list(" ".join(s.split()))        # one space between words (a write pointer does this in place)
    def flip(lo, hi):
        while lo < hi:
            chars[lo], chars[hi] = chars[hi], chars[lo]
            lo, hi = lo + 1, hi - 1
    flip(0, len(chars) - 1)                  # "the sky" -> "yks eht": right order, backwards words
    start = 0
    for i in range(len(chars) + 1):
        if i == len(chars) or chars[i] == " ":
            flip(start, i - 1)               # spell this word forwards again
            start = i + 1
    return "".join(chars)


def zigzag(s, num_rows):
    if num_rows == 1 or num_rows >= len(s):
        return s
    rows, r, step = [[] for _ in range(num_rows)], 0, 1
    for ch in s:
        rows[r].append(ch)
        if r == 0:
            step = 1                         # bounce off the top row
        elif r == num_rows - 1:
            step = -1                        # bounce off the bottom row
        r += step
    return "".join("".join(row) for row in rows)


print(min_remove_to_make_valid("lee(t(c)o)de)"), repr(min_remove_to_make_valid("))((")))   # lee(t(c)o)de ''
print(reverse_words("  the sky   is blue "), zigzag("PAYPALISHIRING", 3))                   # blue is sky the PAHNAPLSIIGYIR

**Try it**
- Print `open_idx` after the loop for `"(a(b(c)d)"`: `[0]`. Only the first `(` never found a partner, so the answer is `'a(b(c)d)'`.
- Replace `chars[i] = ""` in the `)` branch with `del chars[i]` and rerun the cell: `IndexError` on `"))(("`, because the saved indices of the `(` now point past the end of the shorter list. On `"a)b)c"` it fails quietly instead, returning `'ab)'`.
- Remove the second loop (the per-word flips) and run `reverse_words("the sky is blue")`: `'eulb si yks eht'`, halfway there.
- Remove the early `return s` (the first `if` in `zigzag`) and run `zigzag("AB", 1)`: `IndexError`. With one row the step never flips, so `r` walks off to row 1.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Valid Number asks whether a text is a number in decimal or scientific notation: `"2"`, `"-.9"`, `"4."` and `"3e+7"` are, while `"e3"`, `"99e2.5"`, `"."` and `"1e"` are not. It keeps three flags instead of a value, because each character is legal or not depending only on what has been seen. A sign stands only at the start or right after the `e`; there is one dot, never inside the exponent; and one `e`, with digits before it. The exponent needs digits of its own, so the `e` resets "digit seen".

<details><summary>Valid Number in code</summary>

```py
def is_number(s):
    seen_digit = seen_dot = seen_exp = False # the whole memory of this state machine
    for i, c in enumerate(s):
        if "0" <= c <= "9":
            seen_digit = True
        elif c in "+-":
            if i > 0 and s[i - 1] not in "eE":   # a sign only at the start or right after e
                return False
        elif c == ".":
            if seen_dot or seen_exp:         # one dot, and never inside the exponent
                return False
            seen_dot = True
        elif c in "eE":
            if seen_exp or not seen_digit:   # one e, with digits before it
                return False
            seen_exp, seen_digit = True, False   # the exponent needs digits of its own
        else:
            return False                     # a space, a letter, anything else
    return seen_digit

# [is_number(t) for t in ["2", "-.9", "4.", "3e+7", "e3", "99e2.5", ".", "1e"]]
#  -> [True, True, True, True, False, False, False, False]
# without the reset of seen_digit at the e, "1e" would pass
```

</details>

Integer to English Words spells a number in English: 12345 is `"Twelve Thousand Three Hundred Forty Five"`. Work in base 1000, because every 3-digit chunk is spelled the same way and then followed by its scale word, Thousand, Million or Billion. Read the chunks lowest first and put each new one in front of the words so far. An all-zero chunk adds nothing, not even its scale word.

In [ ]:
ONES = ["", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine", "Ten", "Eleven",
        "Twelve", "Thirteen", "Fourteen", "Fifteen", "Sixteen", "Seventeen", "Eighteen", "Nineteen"]
TENS = ["", "", "Twenty", "Thirty", "Forty", "Fifty", "Sixty", "Seventy", "Eighty", "Ninety"]

def number_to_words(num):
    if num == 0:
        return "Zero"
    def chunk(n):                            # the words for 1 <= n <= 999
        words = []
        if n >= 100:
            words += [ONES[n // 100], "Hundred"]
            n %= 100
        if n >= 20:
            words.append(TENS[n // 10])
            n %= 10
        if n:
            words.append(ONES[n])            # 1..19 each have their own word
        return words
    out = []
    for scale in ["", "Thousand", "Million", "Billion"]:   # three digits at a time, lowest first
        if num % 1000:                       # an all-zero chunk adds nothing, not even its scale
            out = chunk(num % 1000) + ([scale] if scale else []) + out
        num //= 1000
    return " ".join(out)


print(number_to_words(12345), "|", number_to_words(1000010))   # Twelve Thousand Three Hundred Forty Five | One Million Ten

**Try it**
- Remove the `if num % 1000:` guard and run `number_to_words(1000000)`: `'Billion One Million Thousand'`. Empty chunks must stay silent, scale word included.
- Change `if n >= 20:` to `if n > 20:` and run `number_to_words(20)`: `IndexError`. Twenty now falls through to `ONES[20]`, and `ONES` stops at Nineteen.
- Append each chunk instead, `out = out + chunk(num % 1000) + ...`, and run `number_to_words(12345)`: `'Three Hundred Forty Five Twelve Thousand'`. The chunks arrive lowest first, so each new one belongs in front.

Palindrome Pairs asks for every pair of different words whose concatenation is a palindrome: in `["abcd", "dcba", "lls", "s", "sssll"]`, `"s" + "lls"` and `"lls" + "sssll"` are two of the four. Cut each word in two. If one half is a palindrome, the other half's reverse must be a whole word, and a dict of reversed words finds it with one lookup: O(n·L²) for n words of length L.

<details><summary>Palindrome Pairs in code</summary>

```py
def palindrome_pairs(words):                 # O(n·L²): n words, L + 1 cuts, O(L) per test
    index = {w[::-1]: i for i, w in enumerate(words)}   # reversed word -> its index
    pairs = []
    for i, w in enumerate(words):
        for cut in range(len(w) + 1):        # w = pre + suf
            pre, suf = w[:cut], w[cut:]
            if pre == pre[::-1] and suf in index and index[suf] != i:
                pairs.append([index[suf], i])    # reverse(suf) + pre + suf
            if cut < len(w) and suf == suf[::-1] and pre in index and index[pre] != i:
                pairs.append([i, index[pre]])    # pre + suf + reverse(pre); cut < len(w) avoids doubles
    return pairs

# palindrome_pairs(["abcd", "dcba", "lls", "s", "sssll"])  -> [[1, 0], [0, 1], [3, 2], [2, 4]]
# without "cut < len(w) and", ["abcd", "dcba"] gives every pair twice
```

</details>

The **KMP prefix function** asks, at every position, how much of the start of the string reappears just before it. Longest Happy Prefix asks it once, for the whole string: `"level"` gives `"l"`. `fail[i]` is the length of the longest *border* of `s[:i + 1]`, a prefix that is also a suffix and is *proper*, shorter than the whole string. When the next letter does not extend the current border, fall back to the border of the border, `fail[k - 1]`, instead of starting over:

```text
s = a a b a a a b        fail so far = [0, 1, 0, 1, 2]
i=5 ('a'): extend the border "aa"? its next letter s[2] = 'b' != 'a'
           fall back to the border of "aa", which is "a": s[1] = 'a' == 'a'  ->  "aa", fail[5] = 2
i=6 ('b'): extend "aa": s[2] = 'b' == 'b'  ->  "aab", fail[6] = 3
```

Each fall-back pays for an earlier step forward, so the table costs O(n) and a search O(n + m). Longest Happy Prefix is `fail[-1]`. Shortest Palindrome adds the fewest letters in front of `s`, `"aacecaaa"` → `"aaacecaaa"`, so it needs the longest palindromic *prefix*: the border of `s + "#" + reverse(s)`. `kmp_find` returns every start of a pattern in a text; it glues the two with `"\0"`, a character in neither string, so no border can cross it.

In [ ]:
def prefix_function(s):
    fail = [0] * len(s)                      # fail[i] = longest proper border of s[:i + 1]
    for i in range(1, len(s)):
        k = fail[i - 1]                      # the border we hope to extend by s[i]
        while k and s[i] != s[k]:
            k = fail[k - 1]                  # fall back to the next shorter border
        if s[i] == s[k]:
            k += 1
        fail[i] = k
    return fail


def longest_happy_prefix(s):                 # the longest proper prefix that is also a suffix
    return s[:prefix_function(s)[-1]] if s else ""


def shortest_palindrome(s):                  # letters may only be added in front
    keep = prefix_function(s + "#" + s[::-1])[-1]   # the longest palindromic prefix of s
    return s[keep:][::-1] + s


def kmp_find(text, pattern):                 # every start of pattern in text, overlaps too
    m, fail = len(pattern), prefix_function(pattern + "\0" + text)
    return [i - 2 * m for i, k in enumerate(fail) if k == m]


print(prefix_function("aabaaab"))                                      # [0, 1, 0, 1, 2, 2, 3]
print(longest_happy_prefix("ababab"), longest_happy_prefix("level"))   # abab l
print(shortest_palindrome("aacecaaa"), shortest_palindrome("abcd"))    # aaacecaaa dcbabcd
print(kmp_find("aabaaabaab", "aab"))                                   # [0, 4, 7]

**Try it**
- Replace `k = fail[k - 1]` with `k = 0` (start over on a mismatch) and rerun: `prefix_function("aabaaab")` is `[0, 1, 0, 1, 2, 1, 0]`. The borders `"aa"` and `"aab"` at the end are missed.
- Remove the `"#"` in `shortest_palindrome` and run it on `"aaba"`: `'aaba'`, which is not a palindrome. The border of `"aabaabaa"` is 5, longer than `s`; the separator caps it at `len(s)`. The right answer is `'abaaba'`.
- Run `kmp_find("aaaa", "aa")`: `[0, 1, 2]`. Overlapping matches are found because a full match keeps its border.

Longest Duplicate Substring asks for the longest substring that occurs at least twice, overlaps allowed: `"banana"` gives `"ana"`. It stacks two ideas. If a substring of length L occurs twice, so does its prefix of length L − 1, so "a repeat of length L exists" is true up to the answer and false after it: binary search the length, as in [Binary Search](05_Binary_Search_Sorting.ipynb#s09).

To test one length in O(n), read each window as a number in base B, modulo a big prime, and *roll* it one step in O(1): in base 10 the window `"123"` becomes `"234"` as 123 · 10 − 1 · 1000 + 4 = 234. Equal strings have equal hashes, and an equal hash is confirmed by comparing letters, because different strings can share one: collisions cost time, never correctness. The whole search is O(n log n) expected.

In Java or C++, add the modulus back after the subtraction; Python's `%` never returns a negative number. The rolling hash itself is coded in `practice/simple/basics/strings/04_rabin_karp_rolling_hash.py`, with a tiny modulus that makes the collisions visible.

Text Justification lays words out in lines of exactly `width` characters with straight edges: at width 16, `["This", "is", "an", "example", "of", "text", "justification."]` becomes three lines, the first `'This    is    an'`. There is no trick, just two jobs done cleanly in O(total characters). Pack as many words as fit on the line, greedily, and spread the spare spaces with one `divmod`: every gap gets the quotient, and the leftmost `remainder` gaps get one more. The last line, and a line with one word, are left-justified.

In [ ]:
def full_justify(words, width):
    lines, i = [], 0
    while i < len(words):
        j, used = i + 1, len(words[i])       # the line is words[i:j]; used = its letters + 1 space per gap
        while j < len(words) and used + 1 + len(words[j]) <= width:
            used += 1 + len(words[j])
            j += 1
        line, gaps = words[i:j], j - i - 1
        if j == len(words) or gaps == 0:     # last line or a single word: left-justify
            text = " ".join(line)
            lines.append(text + " " * (width - len(text)))
        else:
            q, r = divmod(width - sum(map(len, line)), gaps)
            pieces = []
            for k in range(gaps):            # gap k gets q spaces, plus one if k < r
                pieces.append(line[k] + " " * (q + (1 if k < r else 0)))
            lines.append("".join(pieces) + line[-1])
        i = j
    return lines


print(full_justify(["This", "is", "an", "example", "of", "text", "justification."], 16))
# ['This    is    an', 'example  of text', 'justification.  ']

**Try it**
- Work out the second line by hand: `"example"`, `"of"`, `"text"` hold 13 letters, so 3 spaces go into 2 gaps: `divmod(3, 2)` is `(1, 1)`, and the left gap gets the extra one.
- Change `<=` to `<` in the packing loop and run `full_justify(["ab", "cd"], 5)`: `['ab   ', 'cd   ']` instead of `['ab cd']`. A line that fits exactly is legal.
- Remove `j == len(words) or` and run `full_justify(["What", "must", "be", "acknowledgment", "shall", "be"], 16)`: the last line becomes `'shall         be'` (nine spaces in one gap) instead of `'shall be        '`.

### Say it in the interview

> "Two questions first: if two separators are adjacent, do you want an empty string between them, like `split(",")`, or should a run count as one, like `split()`? And can the separator be longer than one character? I'll walk the string once with an index and remember where the current piece started."

> "When the separator starts at `i` I emit `s[start:i]`, which may be empty, jump past the whole separator, and start the next piece there. After the loop I emit the tail; that line makes `"a,"` give `["a", ""]` and `""` give `[""]`. Each position compares up to m characters, so O(n·m), which is O(n) for one character, plus O(n) for the output. Using `find` and slicing off the remainder copies the rest of the string per piece, which is O(n²) at worst."

For any string problem, ask: which alphabet, case-sensitive or not, can it be empty? Then say the edge cases *before* coding (empty string, separator at both ends, two in a row) and point at the final `append` after the loop: it is the line most people forget. For a parser, name the phases out loud ("spaces, then one sign, then digits") and write one `while` per phase.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Compare Version Numbers | `strings/compare_version_numbers.py` | two cursors parse one field each as an int; a used-up string reads as 0 |
| Group Shifted Strings | `strings/group_shifted_strings.py` | key = gaps between neighbours mod 26; a shift doesn't change the gaps |
| Implement String Split (warm-up) | `strings/implement_split.py` | cursor + piece start; emit at each separator, jump past it, emit the tail after the loop |
| Integer to English Words | `strings/integer_to_english_words.py` | chunks of three digits from the right, each spelled alike plus Thousand/Million/Billion; skip zero chunks |
| Longest Duplicate Substring | `strings/longest_duplicate_substring.py` | binary search the length (repeats are monotone); a rolling hash tests one length in O(n) |
| Longest Happy Prefix | `strings/longest_happy_prefix.py` | KMP prefix function; the answer is `s[:fail[-1]]` |
| Longest Palindromic Substring | `strings/longest_palindromic_substring.py` · `practice/simple/50_longest_palindromic_substring.py` | expand around all 2n − 1 centers; the palindrome is `s[lo + 1:hi]` when the loop stops |
| Minimum Remove to Make Valid Parentheses | `strings/minimum_remove_to_make_valid_parentheses.py` | stack of open indices; blank unmatched `)` at once and leftover `(` at the end; join once |
| Palindrome Pairs | `strings/palindrome_pairs.py` | cut each word: one half a palindrome, the other half's reverse found in a dict of reversed words |
| Reverse Words in a String | `strings/reverse_words_in_a_string.py` | squeeze the spaces, reverse the whole list, then reverse each word back |
| Shortest Palindrome | `strings/shortest_palindrome.py` | longest palindromic prefix = border of `s + "#" + reverse(s)`; prepend the rest reversed |
| Simplify Path | `strings/simplify_path.py` | split on "/"; a stack of names: push a name, pop on "..", skip "" and "." ([Stacks & Queues](04_Stacks_Queues_Linked_Lists.ipynb#s07)) |
| String to Integer (atoi) | `strings/string_to_integer_atoi.py` | phases spaces → sign → digits; `num = num * 10 + d`; clamp the moment it overflows |
| Text Justification | `strings/text_justification.py` | greedy packing; `divmod(spaces, gaps)` with extras on the left; last line left-justified |
| Valid Anagram | `strings/valid_anagram.py` | one 26-count array: +1 for s, −1 for t; anagrams iff all counts are 0 (lowercase only: ask) |
| Valid Number | `strings/valid_number.py` | three flags (digit, dot, exponent); a sign only at the start or after e; e resets "digit seen" |
| Zigzag Conversion | `strings/zigzag_conversion.py` | only the row matters: a row index bouncing between the edges, one list per row |

### Self-check

1. Why does `split_on_char` append once more after the loop, and what does that give for `""` and for `"a,"`?
<details><summary>Answer</summary>The last piece has no separator after it, so nothing inside the loop ever closes it. For <code>""</code> it emits the empty piece: <code>[""]</code>. For <code>"a,"</code> it emits the empty tail: <code>["a", ""]</code>. Both match Python's <code>str.split</code>.</details>

2. `" a  b ".split(" ")` or `" a  b ".split()`: which one did the interviewer mean, and how would each change your code?
<details><summary>Answer</summary><code>split(" ")</code> keeps an empty piece between adjacent spaces and at both ends: <code>['', 'a', '', 'b', '']</code>; that is <code>split_on_char</code>. <code>split()</code> treats a run of spaces as one gap and returns no empty words: <code>['a', 'b']</code>; that is <code>split_words</code>, with a phase that skips the gap and a check that drops the empty word a trailing gap reads.</details>

3. The KMP table has a `while` inside a `for`. Why is building it still O(n)?
<details><summary>Answer</summary><code>k</code> grows by at most 1 per character, so it grows at most n times in total, and every turn of the <code>while</code> makes it smaller. It cannot shrink more often than it grew, so all the <code>while</code> turns together cost at most n.</details>

4. In Longest Duplicate Substring, why may you binary search on the length, and why compare letters after a hash hit?
<details><summary>Answer</summary>If a substring of length L occurs twice, its first L − 1 letters also occur twice, so feasibility is true up to the answer and false after it: monotone, so binary search works. Two different strings can have the same hash (a collision), so a hash hit is only a candidate until the letters agree.</details>